## Cell 0. 노트북 개요

목적:
- 이 노트북은 건설산업 조기경보시스템(EWS)을 위한 건설경기 관련 언론보도 수집, 원문 검증, 위험 이벤트 판단, SQLite/Chroma/Excel 저장 PoC를 셀 순서대로 실행할 수 있게 정리한다.

입력값:
- 하단 셀에서 입력하는 사용자 자연어 질의
- NVIDIA_API_KEY

출력값:
- 자연어 응답
- SQLite DB
- Chroma Vector DB
- Excel 검토 결과 파일

다음 셀과의 연결 관계:
- Cell 1부터 패키지 설치, import, API Key 설정, 함수 및 Agent 정의를 차례로 실행한 뒤 Cell 22 또는 Cell 27에서 질의를 실행한다.

사용자가 수정해도 되는 부분:
- 하단의 자연어 질의
- TEST_QUERIES와 QUERY_INDEX
- CONFIG 생성 함수 안의 후보 수, sleep, LLM 호출 수

주의사항:
- Gemini 관련 코드는 사용하지 않는다.
- LLM은 NVIDIA NIM OpenAI-compatible API만 사용한다.
- 기사 전문 전체는 Vector DB에 저장하지 않고, URL/메타데이터/요약/위험 이벤트/판단 근거 중심으로 저장한다.


## Cell 1. 패키지 설치

목적:
- Colab 런타임에 필요한 Python 패키지를 설치한다.

입력값:
- 없음

출력값:
- 설치된 패키지

다음 셀과의 연결 관계:
- Cell 2에서 이 패키지들을 import한다.

사용자가 수정해도 되는 부분:
- 패키지 버전을 고정하고 싶으면 `==버전`을 추가할 수 있다.

주의사항:
- `google-genai`와 Gemini 관련 패키지는 설치하지 않는다.


In [ ]:
# =========================================
# Cell 1. 패키지 설치
# =========================================

!pip -q install langgraph feedparser requests beautifulsoup4 lxml pandas openpyxl chromadb sentence-transformers
!pip -q install googlenewsdecoder trafilatura ftfy openai

print("패키지 설치 완료")


## Cell 2. import 및 warning 설정

목적:
- 전체 Agent 파이프라인에서 사용할 표준 라이브러리와 외부 패키지를 로드한다.

입력값:
- Cell 1에서 설치한 패키지

출력값:
- import된 모듈

다음 셀과의 연결 관계:
- Cell 3에서 NVIDIA_API_KEY를 설정하고, 이후 모든 함수 정의 셀에서 여기서 import한 모듈을 사용한다.

사용자가 수정해도 되는 부분:
- 로깅 레벨
- warning 출력 여부

주의사항:
- Gemini 관련 import를 추가하지 않는다.


In [1]:
# =========================================
# Cell 2. import 및 warning 설정
# =========================================

import os
import re
import json
import time
import sqlite3
import hashlib
import logging
import warnings
from datetime import datetime, timedelta
from typing import TypedDict, List, Dict, Any, Optional
from urllib.parse import quote_plus, urlparse

import pandas as pd
import requests
import feedparser
from bs4 import BeautifulSoup

import chromadb
from chromadb.utils import embedding_functions

import trafilatura
from ftfy import fix_text
from googlenewsdecoder import gnewsdecoder
from openai import OpenAI

from langgraph.graph import StateGraph, END

warnings.filterwarnings("ignore")
logging.getLogger("trafilatura").setLevel(logging.ERROR)
logging.getLogger("urllib3").setLevel(logging.ERROR)
logging.getLogger("chromadb").setLevel(logging.ERROR)
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)

print("import 완료")


import 완료


## Cell 3. NVIDIA NIM API Key 설정

목적:
- NVIDIA NIM API 호출에 사용할 `NVIDIA_API_KEY`를 안전하게 입력받는다.

입력값:
- 사용자가 입력하는 NVIDIA API Key

출력값:
- 환경변수 `NVIDIA_API_KEY`
- `USE_NVIDIA_NIM`

다음 셀과의 연결 관계:
- Cell 5의 자연어 질의 NIM 파서와 Cell 15의 LLM 판단 Agent가 이 값을 사용한다.

사용자가 수정해도 되는 부분:
- 이미 Colab Secret 또는 환경변수에 키가 있다면 입력 없이 넘어가도 된다.

주의사항:
- API Key를 노트북 본문에 하드코딩하지 않는다.
- 키가 없으면 NIM 호출 단계는 규칙 기반 fallback으로 동작한다.


In [2]:
# =========================================
# Cell 3. NVIDIA NIM API Key 설정
# =========================================

from getpass import getpass

if not os.getenv("NVIDIA_API_KEY"):
    nvidia_key = getpass("NVIDIA_API_KEY 입력: ").strip()
    if nvidia_key:
        os.environ["NVIDIA_API_KEY"] = nvidia_key

USE_NVIDIA_NIM = bool(os.getenv("NVIDIA_API_KEY"))

print("NVIDIA NIM API Key 설정 확인 완료")
print("USE_NVIDIA_NIM:", USE_NVIDIA_NIM)

if not USE_NVIDIA_NIM:
    print("NVIDIA_API_KEY가 없습니다. 이후 LLM 판단은 규칙 기반 fallback으로 동작합니다.")


NVIDIA_API_KEY 입력: ··········
NVIDIA NIM API Key 설정 확인 완료
USE_NVIDIA_NIM: True


## Cell 4. 공통 유틸 함수

목적:
- ID 생성, 텍스트 정규화, 한글 인코딩 복구, URL 도메인 검증, 날짜 처리, JSON 파싱, HTML 디코딩 공통 함수를 정의한다.

입력값:
- 문자열, URL, requests Response, 날짜 문자열

출력값:
- 정제된 텍스트
- URL 도메인
- Google News RSS URL
- 안전하게 파싱된 JSON dict

다음 셀과의 연결 관계:
- Cell 5 이후 모든 Agent에서 공통으로 사용한다.

사용자가 수정해도 되는 부분:
- `decode_response_html`의 인코딩 후보
- `get_after_before_dates`의 날짜 검색 범위

주의사항:
- `clean_text`는 title, author, description, body_preview, LLM 결과, Chroma 문서, Excel 저장값에 모두 적용한다.
- `ì`, `ë`, `í` 형태의 mojibake를 방지하기 위해 `ftfy`와 수동 복구를 함께 사용한다.


In [3]:
# =========================================
# Cell 4. 공통 유틸 함수
# =========================================

def make_id(value, prefix=""):
    digest = hashlib.md5(str(value).encode("utf-8")).hexdigest()
    return f"{prefix}_{digest}" if prefix else digest


def normalize_text(x):
    if x is None:
        return ""
    return re.sub(r"\s+", " ", str(x)).strip()


def fix_mojibake_text(text):
    """
    한글이 UTF-8이 아닌 latin1/cp1252처럼 잘못 해석되어
    'ìí', 'ë°°ë¯¼'처럼 깨지는 경우를 최대한 복구한다.
    """
    if text is None:
        return ""

    text = str(text)

    try:
        fixed = fix_text(text)
        if fixed:
            text = fixed
    except Exception:
        pass

    mojibake_markers = [
        "ì", "í", "ë", "ê", "ã", "Â", "Ã", "¢", "€", "œ",
        "\x80", "\x81", "\x82", "\x83", "\x84", "\x85", "\x86", "\x87",
        "\x88", "\x89", "\x8a", "\x8b", "\x8c", "\x8d", "\x8e", "\x8f",
        "\x90", "\x91", "\x92", "\x93", "\x94", "\x95", "\x96", "\x97",
        "\x98", "\x99", "\x9a", "\x9b", "\x9c", "\x9d", "\x9e", "\x9f"
    ]

    if not any(marker in text for marker in mojibake_markers):
        return text

    for enc in ["latin1", "cp1252"]:
        try:
            fixed = text.encode(enc, errors="ignore").decode("utf-8", errors="ignore")
            fixed = fix_text(fixed)
            if fixed and len(fixed.strip()) > 0:
                return fixed
        except Exception:
            continue

    return text


def clean_text(text):
    return normalize_text(fix_mojibake_text(text))


def extract_domain(url):
    if not isinstance(url, str) or not url.strip():
        return ""
    try:
        return urlparse(url).netloc.lower().replace("www.", "")
    except Exception:
        return ""


def is_domain_allowed(url, allowed_domains):
    domain = extract_domain(url)
    return any(
        domain == d.lower().replace("www.", "") or domain.endswith("." + d.lower().replace("www.", ""))
        for d in allowed_domains
    )


def find_outlet_config(outlet_name, outlets):
    for outlet in outlets:
        if outlet["outlet_name"] == outlet_name:
            return outlet
    return None


def get_after_before_dates(target_date: str):
    # Google News RSS의 before는 관례적으로 종료일처럼 쓰이므로 기준일 다음날을 넣는다.
    dt = pd.to_datetime(target_date).date()
    after_date = dt - timedelta(days=1)
    before_date = dt + timedelta(days=1)
    return after_date.isoformat(), before_date.isoformat()


def build_google_news_rss_url(query, hl="ko", gl="KR", ceid="KR:ko"):
    encoded_query = quote_plus(query)
    return f"https://news.google.com/rss/search?q={encoded_query}&hl={hl}&gl={gl}&ceid={ceid}"


def safe_json_loads(text: str) -> dict:
    if not text:
        return {}

    text = clean_text(text.strip())
    text = re.sub(r"^```json", "", text).strip()
    text = re.sub(r"^```", "", text).strip()
    text = re.sub(r"```$", "", text).strip()

    try:
        return json.loads(text)
    except Exception:
        pass

    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except Exception:
            return {}

    return {}


def decode_response_html(res):
    """
    requests Response에서 HTML을 안전하게 디코딩한다.
    한국어 언론사 페이지는 utf-8, cp949, euc-kr이 섞일 수 있으므로 후보를 순서대로 시도한다.
    """
    content_type = res.headers.get("Content-Type", "").lower()
    encodings = []

    charset_match = re.search(r"charset=([\w\-]+)", content_type)
    if charset_match:
        encodings.append(charset_match.group(1))

    encodings.extend(["utf-8", "cp949", "euc-kr"])

    if getattr(res, "encoding", None):
        encodings.append(res.encoding)

    seen = set()
    encodings = [e for e in encodings if e and not (e.lower() in seen or seen.add(e.lower()))]

    for enc in encodings:
        try:
            decoded = res.content.decode(enc, errors="replace")
            return fix_mojibake_text(decoded)
        except Exception:
            continue

    return fix_mojibake_text(res.text)


def extract_date_yyyy_mm_dd(date_text: str) -> str:
    if not date_text:
        return ""

    date_text = clean_text(date_text)

    try:
        dt = pd.to_datetime(date_text, errors="coerce")
        if pd.notna(dt):
            return dt.date().isoformat()
    except Exception:
        pass

    patterns = [
        r"(\d{4})[-./](\d{1,2})[-./](\d{1,2})",
        r"(\d{4})년\s*(\d{1,2})월\s*(\d{1,2})일",
    ]

    for pat in patterns:
        m = re.search(pat, date_text)
        if m:
            y, mo, d = m.groups()
            return f"{int(y):04d}-{int(mo):02d}-{int(d):02d}"

    return ""


print("공통 유틸 함수 정의 완료")


공통 유틸 함수 정의 완료


## Cell 5. 자연어 질의 Parser 함수 정의

목적:
- 사용자의 자연어 질의를 기준일자, 대상 언론사, 검색 키워드로 구조화한다.

입력값:
- `user_query`
- `use_nim_parser`

출력값:
- `target_date`
- `selected_outlets`
- `outlets`
- `keywords`
- parser 종류

다음 셀과의 연결 관계:
- Cell 21의 `build_config_from_user_query`가 이 파싱 결과를 받아 CONFIG를 만든다.

사용자가 수정해도 되는 부분:
- 기본 언론사
- 포괄 질의에서 확장할 키워드
- NIM parser 프롬프트

주의사항:
- NIM 파싱 실패 시 규칙 기반 fallback을 사용한다.
- “오늘”, “금일”, “오늘일자”는 현재 날짜, “어제”는 하루 전으로 해석한다.
- “조중동”은 조선일보, 중앙일보, 동아일보로 해석한다.


In [4]:
# =========================================
# Cell 5. 자연어 질의 Parser 함수 정의
# =========================================

OUTLET_MASTER = {
    "조선일보": {
        "outlet_name": "조선일보",
        "site_domains": ["chosun.com", "biz.chosun.com"],
        "allowed_domains": ["chosun.com", "biz.chosun.com", "realty.chosun.com", "cbiz.chosun.com"]
    },
    "중앙일보": {
        "outlet_name": "중앙일보",
        "site_domains": ["joongang.co.kr"],
        "allowed_domains": ["joongang.co.kr", "www.joongang.co.kr"]
    },
    "동아일보": {
        "outlet_name": "동아일보",
        "site_domains": ["donga.com"],
        "allowed_domains": ["donga.com", "www.donga.com"]
    },
}

DEFAULT_OUTLETS = ["조선일보", "중앙일보", "동아일보"]

DEFAULT_CONSTRUCTION_KEYWORDS = [
    "건설경기", "건설수주", "미분양", "PF 부실", "공사비 상승", "건설투자", "주택시장"
]


def parse_korean_relative_date_rule(user_query: str) -> str:
    user_query = clean_text(user_query)
    today = datetime.now().date()

    if any(k in user_query for k in ["오늘", "금일", "오늘일자", "현재일자"]):
        return today.isoformat()

    if "어제" in user_query:
        return (today - timedelta(days=1)).isoformat()

    m = re.search(r"(\d{4})[-./](\d{1,2})[-./](\d{1,2})", user_query)
    if m:
        y, mo, d = m.groups()
        return f"{int(y):04d}-{int(mo):02d}-{int(d):02d}"

    m = re.search(r"(\d{4})년\s*(\d{1,2})월\s*(\d{1,2})일", user_query)
    if m:
        y, mo, d = m.groups()
        return f"{int(y):04d}-{int(mo):02d}-{int(d):02d}"

    return today.isoformat()


def parse_outlets_from_query_rule(user_query: str) -> list:
    user_query = clean_text(user_query)
    selected = []

    if "조중동" in user_query:
        return DEFAULT_OUTLETS.copy()

    if "조선" in user_query or "조선일보" in user_query:
        selected.append("조선일보")
    if "중앙" in user_query or "중앙일보" in user_query:
        selected.append("중앙일보")
    if "동아" in user_query or "동아일보" in user_query:
        selected.append("동아일보")

    return selected or DEFAULT_OUTLETS.copy()


def parse_keywords_from_query_rule(user_query: str) -> list:
    user_query = clean_text(user_query)
    keyword_pool = {
        "건설경기": ["건설경기", "건설 경기", "건설업 경기", "체감경기"],
        "건설수주": ["건설수주", "수주", "신규 수주"],
        "미분양": ["미분양", "악성 미분양", "준공 후 미분양"],
        "PF 부실": ["PF 부실", "부동산PF", "PF", "브리지론"],
        "공사비 상승": ["공사비 상승", "공사비", "자재비", "인건비"],
        "착공 지연": ["착공", "인허가", "공급 지연"],
        "주택시장": ["주택시장", "청약", "분양", "주택공급"],
        "건설투자": ["건설투자", "투자 부진", "건설 투자"],
    }

    selected = []
    for canonical, variants in keyword_pool.items():
        if any(v.lower() in user_query.lower() for v in variants):
            selected.append(canonical)

    broad_query = any(k in user_query for k in ["건설경기 관련", "건설 경기 관련", "건설 관련", "건설경기"])
    if broad_query:
        for k in DEFAULT_CONSTRUCTION_KEYWORDS:
            if k not in selected:
                selected.append(k)

    return selected[:7] if selected else DEFAULT_CONSTRUCTION_KEYWORDS.copy()


def build_outlet_configs(selected_outlet_names: list) -> list:
    return [OUTLET_MASTER[name] for name in selected_outlet_names if name in OUTLET_MASTER]


def parse_user_news_query_by_rule(user_query: str) -> dict:
    target_date = parse_korean_relative_date_rule(user_query)
    selected_outlets = parse_outlets_from_query_rule(user_query)
    keywords = parse_keywords_from_query_rule(user_query)
    outlets = build_outlet_configs(selected_outlets)

    return {
        "parser": "rule_fallback",
        "original_query": clean_text(user_query),
        "target_date": target_date,
        "selected_outlets": selected_outlets,
        "outlets": outlets,
        "keywords": keywords,
        "task_type": "news_collection_and_verification",
        "need_sqlite": True,
        "need_chroma": True,
        "need_excel": True
    }


def parse_user_news_query_by_nim(user_query: str) -> dict:
    if not USE_NVIDIA_NIM:
        raise RuntimeError("NVIDIA_API_KEY가 없어 NIM parser를 사용할 수 없습니다.")

    today = datetime.now().date().isoformat()
    prompt = f"""
너는 건설산업 조기경보시스템(EWS)을 위한 언론보도 수집 조건 생성 Agent다.

사용자의 자연어 질의를 아래 JSON 형식으로 변환하라.

[현재 날짜]
{today}

[사용자 질의]
{user_query}

[지원 언론사]
- 조선일보
- 중앙일보
- 동아일보

[지원 키워드 후보]
- 건설경기
- 건설수주
- 미분양
- PF 부실
- 공사비 상승
- 건설투자
- 주택시장
- 착공 지연

[해석 규칙]
1. "오늘", "금일", "오늘일자"는 현재 날짜 {today}로 해석한다.
2. "어제"는 현재 날짜의 하루 전으로 해석한다.
3. "조중동"은 ["조선일보", "중앙일보", "동아일보"]로 해석한다.
4. 언론사가 명시되지 않으면 ["조선일보", "중앙일보", "동아일보"]를 기본값으로 둔다.
5. "건설경기 관련"처럼 포괄 질의이면 건설경기, 건설수주, 미분양, PF 부실, 공사비 상승, 건설투자, 주택시장을 포함한다.
6. 출력은 JSON만 작성한다.

{{
  "target_date": "YYYY-MM-DD",
  "selected_outlets": ["조선일보", "중앙일보", "동아일보"],
  "keywords": ["건설경기", "건설수주", "미분양"],
  "task_type": "news_collection_and_verification"
}}
"""

    client = OpenAI(
        base_url="https://integrate.api.nvidia.com/v1",
        api_key=os.getenv("NVIDIA_API_KEY")
    )

    response = client.chat.completions.create(
        model="meta/llama-3.1-70b-instruct",
        messages=[
            {"role": "system", "content": "반드시 JSON만 출력한다."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.0,
        max_tokens=600
    )

    data = safe_json_loads(response.choices[0].message.content)
    if not data:
        raise RuntimeError("NIM parser JSON 파싱 실패")

    selected_outlets = [x for x in data.get("selected_outlets", []) if x in OUTLET_MASTER] or DEFAULT_OUTLETS.copy()
    keywords = [clean_text(x) for x in data.get("keywords", []) if clean_text(x)] or DEFAULT_CONSTRUCTION_KEYWORDS.copy()
    target_date = data.get("target_date") or parse_korean_relative_date_rule(user_query)

    return {
        "parser": "nvidia_nim",
        "original_query": clean_text(user_query),
        "target_date": target_date,
        "selected_outlets": selected_outlets,
        "outlets": build_outlet_configs(selected_outlets),
        "keywords": keywords[:7],
        "task_type": data.get("task_type", "news_collection_and_verification"),
        "need_sqlite": True,
        "need_chroma": True,
        "need_excel": True
    }


def parse_user_news_query(user_query: str, use_nim_parser: bool = True) -> dict:
    if use_nim_parser:
        try:
            return parse_user_news_query_by_nim(user_query)
        except Exception as e:
            print("NVIDIA NIM 질의 파싱 실패. 규칙 기반 fallback을 사용합니다:", e)
    return parse_user_news_query_by_rule(user_query)


print("자연어 질의 Parser 함수 정의 완료")


자연어 질의 Parser 함수 정의 완료


## Cell 6. 기사 메타데이터 및 본문 추출 함수 정의

목적:
- 언론사 HTML에서 제목, 작성자, 발행일, 설명, canonical URL, 본문 preview를 추출한다.

입력값:
- HTML 문자열
- 기사 URL

출력값:
- 기사 메타데이터 dict
- 정제된 본문 문자열

다음 셀과의 연결 관계:
- Cell 13 ArticleVerificationAgent가 이 함수를 사용한다.

사용자가 수정해도 되는 부분:
- 메타 태그 후보
- 본문 selector 후보
- 본문 존재 기준 길이

주의사항:
- 기사 전문은 검증용으로만 사용하고 Vector DB에는 전문 전체를 저장하지 않는다.
- 추출 결과는 모두 `clean_text`로 한글 깨짐을 복구한다.


In [5]:
# =========================================
# Cell 6. 기사 메타데이터 및 본문 추출 함수 정의
# =========================================

def extract_meta_content(soup, keys):
    for key in keys:
        for attr in ["property", "name", "itemprop"]:
            tag = soup.find("meta", attrs={attr: key})
            if tag and tag.get("content"):
                return clean_text(tag.get("content"))
    return ""


def extract_article_metadata(html_text, url):
    html_text = fix_mojibake_text(html_text)
    soup = BeautifulSoup(html_text, "lxml")

    title = extract_meta_content(soup, ["og:title", "twitter:title", "headline"])
    if not title and soup.title:
        title = clean_text(soup.title.get_text())

    author = extract_meta_content(soup, [
        "author", "article:author", "dable:author", "byl",
        "articleAuthor", "parsely-author", "news:author"
    ])

    published_at = extract_meta_content(soup, [
        "article:published_time", "article:modified_time", "date", "pubdate",
        "publishdate", "datePublished", "og:regDate", "parsely-pub-date"
    ])

    description = extract_meta_content(soup, ["og:description", "description", "twitter:description"])

    canonical = ""
    canonical_tag = soup.find("link", rel="canonical")
    if canonical_tag and canonical_tag.get("href"):
        canonical = clean_text(canonical_tag.get("href"))

    return {
        "title": clean_text(title),
        "author": clean_text(author),
        "published_at": clean_text(published_at),
        "description": clean_text(description),
        "canonical_url": clean_text(canonical)
    }


def extract_body_text(html_text, url):
    if not html_text:
        return ""

    html_text = fix_mojibake_text(html_text)

    # trafilatura는 뉴스 본문 추출에 강한 라이브러리다. 실패하면 아래 selector 기반 fallback을 사용한다.
    try:
        extracted = trafilatura.extract(
            html_text,
            include_comments=False,
            include_tables=False,
            favor_precision=True
        )
        extracted = clean_text(extracted)
        if extracted and len(extracted) >= 200:
            return extracted
    except Exception:
        pass

    soup = BeautifulSoup(html_text, "lxml")
    for tag in soup(["script", "style", "noscript", "iframe", "header", "footer", "nav"]):
        tag.decompose()

    candidates = []
    article_tag = soup.find("article")
    if article_tag:
        candidates.append(article_tag.get_text(" ", strip=True))

    selectors = [
        ("div", {"itemprop": "articleBody"}),
        ("section", {"class": re.compile("article|news|body|content|view", re.I)}),
        ("div", {"class": re.compile("article|news|body|content|view|article_body|article-body|article_txt|news_body", re.I)}),
        ("div", {"id": re.compile("article|news|body|content|view|articleBody|article_body", re.I)}),
        ("p", {})
    ]

    for name, attrs in selectors:
        if name == "p":
            p_texts = [p.get_text(" ", strip=True) for p in soup.find_all("p")]
            joined = clean_text(" ".join(p_texts))
            if len(joined) > 100:
                candidates.append(joined)
        else:
            for tag in soup.find_all(name, attrs=attrs):
                txt = clean_text(tag.get_text(" ", strip=True))
                if len(txt) > 100:
                    candidates.append(txt)

    candidates = [clean_text(c) for c in candidates if c and len(clean_text(c)) > 100]
    return clean_text(max(candidates, key=len)) if candidates else ""


print("기사 메타데이터 및 본문 추출 함수 정의 완료")


기사 메타데이터 및 본문 추출 함수 정의 완료


## Cell 7. 위험 이벤트 사전 정의

목적:
- 규칙 기반 위험 이벤트 태깅에 사용할 이벤트 유형과 키워드 사전을 정의한다.

입력값:
- 없음

출력값:
- `RISK_EVENT_RULES`

다음 셀과의 연결 관계:
- Cell 14 RuleRiskEventAgent와 Cell 15 fallback 판단에서 사용한다.

사용자가 수정해도 되는 부분:
- 이벤트 유형
- 이벤트별 키워드 목록

주의사항:
- 이벤트 유형명은 SQLite, Chroma, Excel, 자연어 응답에 그대로 쓰이므로 너무 자주 바꾸지 않는 것이 좋다.


In [6]:
# =========================================
# Cell 7. 위험 이벤트 사전 정의
# =========================================

RISK_EVENT_RULES = {
    "건설수주 감소": ["건설수주", "수주 감소", "수주 부진", "수주 절벽", "신규 수주"],
    "미분양 증가": ["미분양", "악성 미분양", "준공 후 미분양", "계약률 0"],
    "PF 부실": ["PF", "부동산PF", "프로젝트파이낸싱", "브리지론", "PF 부실", "PF 연체"],
    "공사비 상승": ["공사비", "자재비", "인건비", "분양가", "공사비 급등"],
    "착공 지연": ["착공", "인허가", "공급 지연", "사업 지연", "공사 지연"],
    "주택시장 위험": ["주택시장", "청약", "분양", "주택공급", "재개발", "재건축"],
    "건설투자 부진": ["건설투자", "건설경기", "체감경기", "경기 악화", "투자 부진"],
    "기업 재무 건전성": ["상폐", "상장폐지", "유동성", "부채", "워크아웃", "자금난", "재무"]
}

print("위험 이벤트 사전 정의 완료")


위험 이벤트 사전 정의 완료


## Cell 8. SQLite 스키마 생성 함수 정의

목적:
- Agent 실행 결과를 저장할 SQLite 테이블 스키마를 정의한다.

입력값:
- `sqlite_path`

출력값:
- SQLite DB 파일과 7개 테이블

다음 셀과의 연결 관계:
- Cell 17 SQLiteStorageAgent가 저장 전 이 함수를 호출한다.

사용자가 수정해도 되는 부분:
- 컬럼 추가
- 인덱스 추가

주의사항:
- `CREATE TABLE IF NOT EXISTS`를 사용하므로 중복 호출되어도 오류가 나지 않는다.
- 매 실행마다 run_id 기반 신규 DB 경로를 생성한다.


In [7]:
# =========================================
# Cell 8. SQLite 스키마 생성 함수 정의
# =========================================

def create_sqlite_schema(sqlite_path: str):
    os.makedirs(os.path.dirname(sqlite_path), exist_ok=True)
    conn = sqlite3.connect(sqlite_path)
    cur = conn.cursor()

    cur.execute("""
    CREATE TABLE IF NOT EXISTS search_tasks (
        task_id TEXT PRIMARY KEY,
        target_date TEXT,
        outlet_name TEXT,
        site_domain TEXT,
        keyword TEXT,
        query TEXT,
        rss_url TEXT,
        created_at TEXT
    )
    """)

    cur.execute("""
    CREATE TABLE IF NOT EXISTS article_candidates (
        candidate_id TEXT PRIMARY KEY,
        task_id TEXT,
        target_date TEXT,
        outlet_name TEXT,
        site_domain TEXT,
        keyword TEXT,
        candidate_title TEXT,
        candidate_url TEXT,
        rss_link TEXT,
        rss_published TEXT,
        rss_summary TEXT,
        discovered_at TEXT
    )
    """)

    cur.execute("""
    CREATE TABLE IF NOT EXISTS article_access_checks (
        check_id TEXT PRIMARY KEY,
        candidate_id TEXT,
        outlet_name TEXT,
        candidate_url TEXT,
        http_status INTEGER,
        final_url TEXT,
        final_domain TEXT,
        domain_matched INTEGER,
        content_type TEXT,
        access_result TEXT,
        failure_reason TEXT,
        resolver_status TEXT,
        resolver_message TEXT,
        checked_at TEXT
    )
    """)

    cur.execute("""
    CREATE TABLE IF NOT EXISTS article_metadata (
        article_id TEXT PRIMARY KEY,
        candidate_id TEXT,
        outlet_name TEXT,
        title TEXT,
        author TEXT,
        published_at TEXT,
        published_date TEXT,
        description TEXT,
        canonical_url TEXT,
        body_exists INTEGER,
        body_length INTEGER,
        body_preview TEXT,
        date_matched INTEGER,
        article_page_result TEXT,
        metadata_quality TEXT,
        verification_score INTEGER,
        quality_grade TEXT
    )
    """)

    cur.execute("""
    CREATE TABLE IF NOT EXISTS rule_risk_events (
        event_id TEXT PRIMARY KEY,
        article_id TEXT,
        outlet_name TEXT,
        event_type TEXT,
        event_keyword TEXT,
        evidence_text TEXT,
        relevance_result TEXT
    )
    """)

    cur.execute("""
    CREATE TABLE IF NOT EXISTS llm_judgements (
        judgement_id TEXT PRIMARY KEY,
        article_id TEXT,
        outlet_name TEXT,
        relatedness TEXT,
        main_risk_event TEXT,
        risk_events_json TEXT,
        fact_opinion_forecast TEXT,
        event_summary TEXT,
        judgement_reason TEXT,
        llm_model TEXT,
        llm_provider_used TEXT,
        llm_used INTEGER,
        created_at TEXT
    )
    """)

    cur.execute("""
    CREATE TABLE IF NOT EXISTS cross_source_checks (
        cross_id TEXT PRIMARY KEY,
        event_type TEXT,
        outlet_count INTEGER,
        article_count INTEGER,
        cross_source_confirmed INTEGER,
        evidence_summary TEXT
    )
    """)

    conn.commit()
    conn.close()


print("SQLite 스키마 생성 함수 정의 완료")


SQLite 스키마 생성 함수 정의 완료


## Cell 9. LangGraph State 정의

목적:
- Agent 사이에서 전달되는 상태 객체의 key와 초기값을 정의한다.

입력값:
- CONFIG dict

출력값:
- `NewsNIMAgentState`
- `init_state(config)`

다음 셀과의 연결 관계:
- Cell 20 Workflow Compile과 Cell 21 전체 실행 함수에서 사용한다.

사용자가 수정해도 되는 부분:
- 새로운 Agent가 추가될 경우 state key 추가

주의사항:
- 기존 state key 이름을 임의로 바꾸면 Agent 연결이 끊길 수 있다.


In [8]:
# =========================================
# Cell 9. LangGraph State 정의
# =========================================

class NewsNIMAgentState(TypedDict):
    target_date: str
    outlets: List[Dict[str, Any]]
    keywords: List[str]
    hl: str
    gl: str
    ceid: str
    sleep_sec: float
    max_candidates_per_task: int
    max_total_candidates: int
    llm_provider: str
    nvidia_nim_base_url: str
    nvidia_nim_model: str
    llm_sleep_sec: float
    max_llm_calls: int
    stop_llm_on_quota_error: bool
    require_date_match_for_vector: bool
    sqlite_path: str
    chroma_path: str
    excel_path: str
    user_news_query: str
    parsed_outlets: List[str]
    search_tasks: List[Dict[str, Any]]
    candidates: List[Dict[str, Any]]
    access_checked: List[Dict[str, Any]]
    article_verified: List[Dict[str, Any]]
    rule_enriched: List[Dict[str, Any]]
    llm_enriched: List[Dict[str, Any]]
    cross_check_results: List[Dict[str, Any]]
    excel_reflection: Dict[str, Any]
    sqlite_frames: Dict[str, Any]
    saved_article_count: int
    saved_vector_count: int
    logs: List[str]
    errors: List[str]


def init_state(config: dict) -> NewsNIMAgentState:
    return {
        "target_date": config["target_date"],
        "outlets": config["outlets"],
        "keywords": config["keywords"],
        "hl": config["hl"],
        "gl": config["gl"],
        "ceid": config["ceid"],
        "sleep_sec": config["sleep_sec"],
        "max_candidates_per_task": config["max_candidates_per_task"],
        "max_total_candidates": config["max_total_candidates"],
        "llm_provider": config["llm_provider"],
        "nvidia_nim_base_url": config["nvidia_nim_base_url"],
        "nvidia_nim_model": config["nvidia_nim_model"],
        "llm_sleep_sec": config["llm_sleep_sec"],
        "max_llm_calls": config["max_llm_calls"],
        "stop_llm_on_quota_error": config["stop_llm_on_quota_error"],
        "require_date_match_for_vector": config["require_date_match_for_vector"],
        "sqlite_path": config["sqlite_path"],
        "chroma_path": config["chroma_path"],
        "excel_path": config["excel_path"],
        "user_news_query": config.get("user_news_query", ""),
        "parsed_outlets": config.get("parsed_outlets", []),
        "search_tasks": [],
        "candidates": [],
        "access_checked": [],
        "article_verified": [],
        "rule_enriched": [],
        "llm_enriched": [],
        "cross_check_results": [],
        "excel_reflection": {},
        "sqlite_frames": {},
        "saved_article_count": 0,
        "saved_vector_count": 0,
        "logs": [],
        "errors": [],
    }


print("LangGraph State 정의 완료")


LangGraph State 정의 완료


## Cell 10. Agent Node 1: QueryPlanningAgent

목적:
- target_date, outlet, site_domain, keyword를 조합해 Google News RSS 검색 작업을 생성한다.

입력값:
- `state["target_date"]`
- `state["outlets"]`
- `state["keywords"]`

출력값:
- `state["search_tasks"]`

다음 셀과의 연결 관계:
- Cell 11 RSSCandidateAgent가 `search_tasks`의 RSS URL을 수집한다.

사용자가 수정해도 되는 부분:
- Google News 검색 query 템플릿
- after/before 날짜 범위

주의사항:
- `site:` 조건과 `after:`/`before:` 조건을 함께 넣어 언론사와 날짜 범위를 제한한다.


In [9]:
# =========================================
# Cell 10. Agent Node 1: QueryPlanningAgent
# =========================================

def query_planning_agent_node(state: NewsNIMAgentState) -> NewsNIMAgentState:
    tasks = []
    after_date, before_date = get_after_before_dates(state["target_date"])

    for outlet in state["outlets"]:
        outlet_name = outlet["outlet_name"]
        for site_domain in outlet["site_domains"]:
            for keyword in state["keywords"]:
                # Google News RSS 검색 query: 키워드 + 대상 도메인 + 날짜 범위
                query = f'{keyword} site:{site_domain} after:{after_date} before:{before_date}'
                task_id = make_id(f"{state['target_date']}_{outlet_name}_{site_domain}_{keyword}", prefix="task")
                rss_url = build_google_news_rss_url(query, hl=state["hl"], gl=state["gl"], ceid=state["ceid"])

                tasks.append({
                    "task_id": task_id,
                    "target_date": state["target_date"],
                    "outlet_name": outlet_name,
                    "site_domain": site_domain,
                    "keyword": keyword,
                    "query": query,
                    "rss_url": rss_url,
                    "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                })

    state["search_tasks"] = tasks
    state["logs"].append(f"[QueryPlanningAgent] RSS 검색 작업 {len(tasks)}개 생성")
    return state


print("QueryPlanningAgent 정의 완료")


QueryPlanningAgent 정의 완료


## Cell 11. Agent Node 2: RSSCandidateAgent

목적:
- Google News RSS에서 후보 기사 URL을 수집하고 중복 URL을 제거한다.

입력값:
- `state["search_tasks"]`

출력값:
- `state["candidates"]`

다음 셀과의 연결 관계:
- Cell 12 URLAccessAgent가 후보 URL을 원문 URL로 해석하고 접근성을 검증한다.

사용자가 수정해도 되는 부분:
- `max_candidates_per_task`
- `max_total_candidates`
- `sleep_sec`

주의사항:
- Google News RSS link는 중간 URL일 수 있으므로 다음 셀에서 resolver를 거친다.


In [10]:
# =========================================
# Cell 11. Agent Node 2: RSSCandidateAgent
# =========================================

def rss_candidate_agent_node(state: NewsNIMAgentState) -> NewsNIMAgentState:
    candidates = []
    seen_urls = set()

    for task in state["search_tasks"]:
        try:
            feed = feedparser.parse(task["rss_url"])
            entries = feed.entries or []
            used_count = 0

            for entry in entries:
                if used_count >= state["max_candidates_per_task"]:
                    break

                title = clean_text(entry.get("title", ""))
                link = clean_text(entry.get("link", ""))
                published = clean_text(entry.get("published", ""))
                summary = clean_text(entry.get("summary", ""))

                if not link or link in seen_urls:
                    continue

                seen_urls.add(link)
                candidate_id = make_id(f"{task['task_id']}_{link}", prefix="cand")

                candidates.append({
                    "candidate_id": candidate_id,
                    "task_id": task["task_id"],
                    "target_date": task["target_date"],
                    "outlet_name": task["outlet_name"],
                    "site_domain": task["site_domain"],
                    "keyword": task["keyword"],
                    "candidate_title": title,
                    "candidate_url": link,
                    "rss_link": link,
                    "rss_published": published,
                    "rss_summary": summary,
                    "discovered_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                })
                used_count += 1

            state["logs"].append(
                f"[RSSCandidateAgent] {task['outlet_name']}/{task['site_domain']}/{task['keyword']}: "
                f"{len(entries)}건 중 {used_count}건 사용"
            )
            time.sleep(state["sleep_sec"])

        except Exception as e:
            state["errors"].append(f"[RSSCandidateAgent] RSS 수집 실패: {task.get('rss_url')} | {e}")

    state["candidates"] = candidates[:state["max_total_candidates"]]
    state["logs"].append(f"[RSSCandidateAgent] 후보 URL {len(seen_urls)}건 수집, 최종 {len(state['candidates'])}건 사용")
    return state


print("RSSCandidateAgent 정의 완료")


RSSCandidateAgent 정의 완료


## Cell 12. URLAccessAgent: Google News URL Resolver 및 접근성 검증

목적:
- Google News RSS의 중간 URL을 실제 언론사 원문 URL로 변환하고, 허용된 언론사 도메인에 속하는지 검증한다.

입력값:
- `state["candidates"]`

출력값:
- `state["access_checked"]`

다음 셀과의 연결 관계:
- Cell 13 ArticleVerificationAgent가 접근 가능한 HTML에서 기사 여부와 본문을 검증한다.

사용자가 수정해도 되는 부분:
- timeout
- User-Agent
- allowed_domains

주의사항:
- Google News URL 해석 실패 시 access_result는 "원문 URL 미확보"로 기록한다.
- HTML 전문은 후속 기사 검증을 위해 state에 임시 보관하지만, 최종 RAG에는 기사 전문을 저장하지 않는다.


In [11]:
# =========================================
# Cell 12. Agent Node 3: URLAccessAgent
# =========================================

def resolve_google_news_url(candidate_url: str, interval: int = 1) -> dict:
    if not candidate_url:
        return {"status": False, "resolved_url": "", "message": "empty url"}

    try:
        decoded = gnewsdecoder(candidate_url, interval=interval)
        if decoded.get("status") and decoded.get("decoded_url"):
            return {"status": True, "resolved_url": decoded["decoded_url"], "message": "resolved"}
        return {"status": False, "resolved_url": "", "message": decoded.get("message", "decode failed")}
    except Exception as e:
        return {"status": False, "resolved_url": "", "message": str(e)}


def url_access_agent_node(state: NewsNIMAgentState) -> NewsNIMAgentState:
    checked = []

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        ),
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
    }

    for row in state["candidates"]:
        item = dict(row)
        candidate_url = item.get("candidate_url", "")
        outlet_name = item.get("outlet_name", "")
        outlet_cfg = find_outlet_config(outlet_name, state["outlets"])
        allowed_domains = outlet_cfg["allowed_domains"] if outlet_cfg else []

        item["check_id"] = make_id(item.get("candidate_id", "") + "_access", prefix="chk")
        item["checked_at"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        candidate_domain = extract_domain(candidate_url)
        source_url = candidate_url
        resolver_status = "not_needed"
        resolver_message = ""

        # Google News 중간 URL이면 googlenewsdecoder로 원문 URL을 해석한다.
        if "news.google.com" in candidate_domain:
            resolved = resolve_google_news_url(candidate_url, interval=1)
            if resolved["status"]:
                source_url = resolved["resolved_url"]
                resolver_status = "resolved"
                resolver_message = "Google News URL 원문 URL 해석 성공"
            else:
                item.update({
                    "http_status": None,
                    "final_url": candidate_url,
                    "final_domain": candidate_domain,
                    "domain_matched": 0,
                    "content_type": "",
                    "access_result": "원문 URL 미확보",
                    "failure_reason": f"Google News URL 해석 실패: {resolved['message']}",
                    "resolver_status": "failed",
                    "resolver_message": resolved["message"],
                    "html_text": ""
                })
                checked.append(item)
                continue

        try:
            res = requests.get(source_url, headers=headers, timeout=15, allow_redirects=True)
            content_type = res.headers.get("Content-Type", "")
            final_url = res.url
            final_domain = extract_domain(final_url)
            domain_matched = int(is_domain_allowed(final_url, allowed_domains))

            html_text = ""
            if "text/html" in content_type.lower():
                html_text = decode_response_html(res)

            if res.status_code == 200 and domain_matched == 1 and html_text:
                access_result = "접근 가능"
                failure_reason = ""
            elif res.status_code == 200 and domain_matched == 0:
                access_result = "제한"
                failure_reason = f"최종 URL 도메인 불일치: {final_domain}"
            elif res.status_code in [401, 403]:
                access_result = "제한"
                failure_reason = f"접근 제한 HTTP {res.status_code}"
            elif res.status_code in [404, 410]:
                access_result = "실패"
                failure_reason = f"기사 URL 없음 또는 삭제 HTTP {res.status_code}"
            else:
                access_result = "실패"
                failure_reason = f"HTTP {res.status_code}"

            item.update({
                "http_status": res.status_code,
                "final_url": final_url,
                "final_domain": final_domain,
                "domain_matched": domain_matched,
                "content_type": content_type,
                "access_result": access_result,
                "failure_reason": failure_reason,
                "resolver_status": resolver_status,
                "resolver_message": resolver_message,
                "html_text": html_text
            })

        except requests.exceptions.Timeout:
            item.update({
                "http_status": None,
                "final_url": source_url,
                "final_domain": extract_domain(source_url),
                "domain_matched": 0,
                "content_type": "",
                "access_result": "실패",
                "failure_reason": "요청 시간 초과",
                "resolver_status": resolver_status,
                "resolver_message": resolver_message,
                "html_text": ""
            })
        except Exception as e:
            item.update({
                "http_status": None,
                "final_url": source_url,
                "final_domain": extract_domain(source_url),
                "domain_matched": 0,
                "content_type": "",
                "access_result": "실패",
                "failure_reason": f"요청 오류: {str(e)}",
                "resolver_status": resolver_status,
                "resolver_message": resolver_message,
                "html_text": ""
            })

        checked.append(item)
        time.sleep(state.get("sleep_sec", 0.7))

    state["access_checked"] = checked
    result_counts = pd.Series([x.get("access_result", "미분류") for x in checked]).value_counts(dropna=False).to_dict()
    resolver_counts = pd.Series([x.get("resolver_status", "none") for x in checked]).value_counts(dropna=False).to_dict()
    state["logs"].append(f"[URLAccessAgent] URL 접근성 검증 {len(checked)}건 완료 | 접근 결과 {result_counts} | Resolver {resolver_counts}")
    return state


print("URLAccessAgent 정의 완료")


URLAccessAgent 정의 완료


## Cell 13. Agent Node 4: ArticleVerificationAgent

목적:
- 접근 가능한 원문 HTML이 실제 기사인지 검증하고 본문 존재 여부와 메타데이터 품질을 평가한다.

입력값:
- `state["access_checked"]`

출력값:
- `state["article_verified"]`

다음 셀과의 연결 관계:
- Cell 14 RuleRiskEventAgent가 검증된 기사 텍스트를 기반으로 위험 이벤트를 태깅한다.

사용자가 수정해도 되는 부분:
- 본문 존재 기준 길이
- verification_score 계산 방식
- quality_grade 기준

주의사항:
- body_preview만 저장하고 기사 전문 전체는 SQLite/Chroma에 저장하지 않는다.


In [12]:
# =========================================
# Cell 13. Agent Node 4: ArticleVerificationAgent
# =========================================

def article_verification_agent_node(state: NewsNIMAgentState) -> NewsNIMAgentState:
    verified = []

    for row in state["access_checked"]:
        item = dict(row)
        html_text = item.get("html_text", "")
        item["article_id"] = make_id(item.get("candidate_id", "") + "_article", prefix="art")

        if item.get("access_result") != "접근 가능" or not html_text:
            item.update({
                "title": clean_text(item.get("candidate_title", "")),
                "author": "",
                "published_at": "",
                "published_date": "",
                "description": "",
                "canonical_url": "",
                "body_exists": 0,
                "body_length": 0,
                "body_preview": "",
                "date_matched": 0,
                "article_page_result": "접근 실패",
                "metadata_quality": "낮음",
                "verification_score": 0,
                "quality_grade": "낮음"
            })
            verified.append(item)
            continue

        meta = extract_article_metadata(html_text, item.get("final_url", ""))
        body_text = clean_text(extract_body_text(html_text, item.get("final_url", "")))

        title = clean_text(meta.get("title") or item.get("candidate_title", ""))
        author = clean_text(meta.get("author", ""))
        published_at = clean_text(meta.get("published_at", ""))
        published_date = extract_date_yyyy_mm_dd(published_at)
        description = clean_text(meta.get("description", ""))
        canonical_url = clean_text(meta.get("canonical_url", ""))

        body_length = len(body_text)
        body_exists = int(body_length >= 200)
        date_matched = int(published_date == state["target_date"]) if published_date else 0

        if body_exists == 1:
            article_page_result = "실제 기사"
        elif title or description:
            article_page_result = "본문 확인 제한"
        else:
            article_page_result = "기사 아님"

        metadata_score = sum([bool(title), bool(author), bool(published_date), bool(description), body_exists == 1])
        metadata_quality = "높음" if metadata_score >= 4 else ("중간" if metadata_score >= 2 else "낮음")

        verification_score = 0
        verification_score += int(item.get("access_result") == "접근 가능")
        verification_score += int(item.get("domain_matched") == 1)
        verification_score += int(article_page_result in ["실제 기사", "본문 확인 제한"])
        verification_score += int(bool(title))
        verification_score += int(bool(published_date))
        verification_score += int(bool(author))
        verification_score += int(body_exists == 1)
        verification_score += int(date_matched == 1)
        quality_grade = "높음" if verification_score >= 8 else ("중간" if verification_score >= 5 else "낮음")

        item.update({
            "title": title,
            "author": author,
            "published_at": published_at,
            "published_date": published_date,
            "description": description,
            "canonical_url": canonical_url,
            "body_exists": body_exists,
            "body_length": body_length,
            "body_preview": clean_text(body_text[:800]),
            "date_matched": date_matched,
            "article_page_result": article_page_result,
            "metadata_quality": metadata_quality,
            "verification_score": verification_score,
            "quality_grade": quality_grade
        })
        verified.append(item)

    state["article_verified"] = verified
    result_counts = pd.Series([x.get("article_page_result", "미분류") for x in verified]).value_counts(dropna=False).to_dict()
    body_count = sum([x.get("body_exists", 0) for x in verified])
    state["logs"].append(f"[ArticleVerificationAgent] 기사 검증 {len(verified)}건 완료 | 결과 {result_counts} | 본문 확인 {body_count}건")
    return state


print("ArticleVerificationAgent 정의 완료")


ArticleVerificationAgent 정의 완료


## Cell 14. Agent Node 5: RuleRiskEventAgent

목적:
- 제목, 설명, 본문 preview에서 규칙 기반 위험 이벤트를 태깅한다.

입력값:
- `state["article_verified"]`
- `RISK_EVENT_RULES`

출력값:
- `state["rule_enriched"]`

다음 셀과의 연결 관계:
- Cell 15 LLMJudgementAgent가 NIM 호출 조건 판단과 fallback 판단에 이 결과를 사용한다.

사용자가 수정해도 되는 부분:
- evidence_text 길이
- relevance_result 값

주의사항:
- 이 단계는 LLM 호출 전 1차 필터이자 NIM 실패 시 최소 판단 근거다.


In [13]:
# =========================================
# Cell 14. Agent Node 5: RuleRiskEventAgent
# =========================================

def rule_risk_event_agent_node(state: NewsNIMAgentState) -> NewsNIMAgentState:
    enriched = []

    for row in state["article_verified"]:
        item = dict(row)
        evidence_text = clean_text(" ".join([
            item.get("candidate_title", ""),
            item.get("title", ""),
            item.get("description", ""),
            item.get("body_preview", "")
        ]))

        rule_events = []
        for event_type, keywords in RISK_EVENT_RULES.items():
            for kw in keywords:
                if kw.lower() in evidence_text.lower():
                    event_id = make_id(f"{item.get('article_id')}_{event_type}_{kw}", prefix="rule_evt")
                    rule_events.append({
                        "event_id": event_id,
                        "article_id": item.get("article_id"),
                        "outlet_name": item.get("outlet_name"),
                        "event_type": event_type,
                        "event_keyword": kw,
                        "evidence_text": clean_text(evidence_text[:500]),
                        "relevance_result": "관련 후보"
                    })
                    break

        item["rule_risk_events"] = rule_events
        enriched.append(item)

    state["rule_enriched"] = enriched
    total_events = sum(len(x.get("rule_risk_events", [])) for x in enriched)
    state["logs"].append(f"[RuleRiskEventAgent] 규칙 기반 위험 이벤트 태깅 {len(enriched)}건 완료 | 이벤트 {total_events}건")
    return state


print("RuleRiskEventAgent 정의 완료")


RuleRiskEventAgent 정의 완료


## Cell 15. Agent Node 6: NVIDIA NIM LLMJudgementAgent

목적:
- NVIDIA NIM API로 기사 관련성, 주요 위험 이벤트, 사실/의견/전망, 요약, 판단 근거를 생성한다.

입력값:
- `state["rule_enriched"]`
- `state["nvidia_nim_model"]`
- `state["llm_sleep_sec"]`
- `state["max_llm_calls"]`

출력값:
- `state["llm_enriched"]`

다음 셀과의 연결 관계:
- Cell 16 CrossSourceCheckAgent가 LLM 또는 fallback 판단 결과를 기준으로 교차확인한다.

사용자가 수정해도 되는 부분:
- NIM model
- `max_llm_calls`
- `llm_sleep_sec`
- 프롬프트

주의사항:
- 전체 후보 기사에 LLM을 전수 호출하지 않는다.
- 접근 가능, 도메인 일치, 실제 기사/본문 확인 제한, 위험 신호가 있는 기사만 최대 15건 호출한다.
- rate limit 또는 quota 오류가 발생하면 규칙 기반 fallback으로 전환한다.


In [14]:
# =========================================
# Cell 15. Agent Node 6: NVIDIA NIM LLMJudgementAgent
# =========================================

def has_risk_signal(article: dict) -> bool:
    text = clean_text(" ".join([
        article.get("candidate_title", ""),
        article.get("title", ""),
        article.get("description", ""),
        article.get("body_preview", "")
    ]))

    if article.get("rule_risk_events"):
        return True

    risk_keywords = [
        "건설경기", "건설수주", "건설투자", "미분양", "PF", "부동산PF",
        "공사비", "착공", "인허가", "주택공급", "분양", "주택시장",
        "부동산 경기", "재개발", "재건축", "유동성", "부채", "워크아웃"
    ]
    return any(k.lower() in text.lower() for k in risk_keywords)


def should_call_llm(article: dict) -> bool:
    final_url = article.get("final_url", "")
    return (
        article.get("access_result") == "접근 가능"
        and article.get("domain_matched") == 1
        and article.get("article_page_result") in ["실제 기사", "본문 확인 제한"]
        and bool(article.get("title") or article.get("candidate_title"))
        and bool(final_url)
        and "news.google.com" not in extract_domain(final_url)
        and has_risk_signal(article)
    )


def build_news_judgement_prompt(article: dict) -> str:
    rule_events = [e.get("event_type", "") for e in article.get("rule_risk_events", []) if e.get("event_type")]
    return f"""
너는 건설산업 조기경보시스템(EWS)을 위한 언론보도 검증 Agent다.

아래 기사가 "건설경기 관련 언론보도"로 활용 가능한지 판단하라.

[판단 기준]
- 관련: 건설수주, 미분양, 공사비, PF 부실, 착공/인허가, 주택공급, 건설투자, 건설사 재무 위험과 직접 관련
- 부분 관련: 개별 건설사, 주거, 재개발/재건축 등 건설경기 간접 신호
- 무관: 건설경기 EWS 판단에 활용하기 어려움

[기사 정보]
언론사: {article.get("outlet_name", "")}
제목: {clean_text(article.get("title") or article.get("candidate_title", ""))}
작성일자: {article.get("published_date", "")}
URL: {article.get("final_url", "")}
설명: {clean_text(article.get("description", ""))}
본문 일부: {clean_text(article.get("body_preview", ""))[:1200]}
규칙 기반 이벤트 후보: {", ".join(rule_events)}

[출력 형식]
반드시 JSON만 출력하라.

{{
  "relatedness": "관련|부분 관련|무관",
  "main_risk_event": "건설수주 감소|미분양 증가|PF 부실|공사비 상승|착공 지연|주택시장 위험|건설투자 부진|기업 재무 건전성|기타",
  "risk_events": ["이벤트1", "이벤트2"],
  "fact_opinion_forecast": "사실|의견|전망|혼합",
  "event_summary": "기사의 건설경기 관련 핵심 내용을 1문장으로 요약",
  "reason": "판단 근거를 1~2문장으로 설명"
}}
""".strip()


def call_nvidia_nim_judgement(article: dict, model_name: str, base_url: str) -> dict:
    if not USE_NVIDIA_NIM:
        return {
            "relatedness": "호출 실패",
            "main_risk_event": "",
            "risk_events": [],
            "fact_opinion_forecast": "",
            "event_summary": "",
            "reason": "NVIDIA_API_KEY 미설정"
        }

    try:
        client = OpenAI(base_url=base_url, api_key=os.getenv("NVIDIA_API_KEY"))
        response = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": "너는 건설산업 EWS 언론보도 검증 Agent다. 반드시 JSON만 출력한다."},
                {"role": "user", "content": build_news_judgement_prompt(article)}
            ],
            temperature=0.1,
            max_tokens=800
        )

        text = response.choices[0].message.content
        data = safe_json_loads(text)
        if not data:
            return {
                "relatedness": "호출 실패",
                "main_risk_event": "",
                "risk_events": [],
                "fact_opinion_forecast": "",
                "event_summary": "",
                "reason": f"NVIDIA NIM JSON 파싱 실패: {clean_text(text)[:300]}"
            }

        return {
            "relatedness": clean_text(data.get("relatedness", "")),
            "main_risk_event": clean_text(data.get("main_risk_event", "")),
            "risk_events": data.get("risk_events", []) if isinstance(data.get("risk_events", []), list) else [],
            "fact_opinion_forecast": clean_text(data.get("fact_opinion_forecast", "")),
            "event_summary": clean_text(data.get("event_summary", "")),
            "reason": clean_text(data.get("reason", ""))
        }
    except Exception as e:
        return {
            "relatedness": "호출 실패",
            "main_risk_event": "",
            "risk_events": [],
            "fact_opinion_forecast": "",
            "event_summary": "",
            "reason": f"NVIDIA NIM 호출 오류: {str(e)}"
        }


def is_quota_error(judgement: dict) -> bool:
    reason_text = str(judgement.get("reason", ""))
    return (
        judgement.get("relatedness") == "호출 실패"
        and any(x in reason_text.lower() for x in ["429", "resource_exhausted", "quota", "rate limit", "too many requests", "throttle", "throttling"])
    )


def build_fallback_judgement(item: dict, quota_exhausted: bool = False) -> dict:
    rule_events = [e["event_type"] for e in item.get("rule_risk_events", []) if e.get("event_type")]
    is_verified_article = (
        item.get("access_result") == "접근 가능"
        and item.get("domain_matched") == 1
        and item.get("article_page_result") in ["실제 기사", "본문 확인 제한"]
        and "news.google.com" not in extract_domain(item.get("final_url", ""))
    )

    if is_verified_article and has_risk_signal(item):
        fallback_relatedness = "관련 후보" if rule_events else "미판정"
        fallback_events = rule_events if rule_events else []
        fallback_main_event = rule_events[0] if rule_events else ""
        fallback_summary = clean_text(item.get("description", "") or item.get("body_preview", ""))[:200]
        fallback_reason = "NVIDIA NIM quota/rate limit 초과 감지 후 규칙 기반 fallback 사용" if quota_exhausted else "NVIDIA NIM 미사용 또는 호출 조건 미충족으로 규칙 기반 fallback 사용"
    else:
        fallback_relatedness = "검증 제외"
        fallback_events = []
        fallback_main_event = ""
        fallback_summary = ""
        fallback_reason = "URL/기사 검증 조건 또는 위험 신호 조건 미충족"

    return {
        "relatedness": fallback_relatedness,
        "main_risk_event": fallback_main_event,
        "risk_events": fallback_events,
        "fact_opinion_forecast": "",
        "event_summary": clean_text(fallback_summary),
        "reason": fallback_reason
    }


def llm_judgement_agent_node(state: NewsNIMAgentState) -> NewsNIMAgentState:
    enriched = []
    llm_attempt_count = 0
    llm_success_count = 0
    quota_exhausted = False

    max_calls = state.get("max_llm_calls", 15)
    stop_on_quota = state.get("stop_llm_on_quota_error", True)

    for row in state["rule_enriched"]:
        item = dict(row)
        should_call = should_call_llm(item) and not quota_exhausted and llm_attempt_count < max_calls

        if should_call:
            judgement = call_nvidia_nim_judgement(item, state["nvidia_nim_model"], state["nvidia_nim_base_url"])
            llm_attempt_count += 1

            if is_quota_error(judgement):
                quota_exhausted = bool(stop_on_quota)
                judgement = build_fallback_judgement(item, quota_exhausted=True)
                llm_used = 0
                provider_used = "fallback_after_nim_quota"
            elif judgement.get("relatedness") == "호출 실패":
                judgement = build_fallback_judgement(item, quota_exhausted=False)
                llm_used = 0
                provider_used = "fallback_after_nim_error"
            else:
                llm_success_count += 1
                llm_used = 1
                provider_used = "nvidia_nim"

            # 무료/공유 endpoint는 throttling 가능성이 있으므로 기본 3초 간격을 둔다.
            time.sleep(state["llm_sleep_sec"])
        else:
            judgement = build_fallback_judgement(item, quota_exhausted=quota_exhausted)
            llm_used = 0
            provider_used = "fallback"

        item["llm_judgement"] = judgement
        item["llm_used"] = llm_used
        item["llm_provider_used"] = provider_used

        verification_score = item.get("verification_score", 0)
        verification_score += int(judgement.get("relatedness") in ["관련", "부분 관련", "관련 후보"])
        verification_score += int(bool(judgement.get("main_risk_event")))
        item["verification_score"] = verification_score
        item["quality_grade"] = "높음" if verification_score >= 8 else ("중간" if verification_score >= 5 else "낮음")

        enriched.append(item)

    state["llm_enriched"] = enriched
    relatedness_counts = pd.Series([x.get("llm_judgement", {}).get("relatedness", "미분류") for x in enriched]).value_counts(dropna=False).to_dict()
    provider_counts = pd.Series([x.get("llm_provider_used", "unknown") for x in enriched]).value_counts(dropna=False).to_dict()
    state["logs"].append(
        f"[LLMJudgementAgent] NVIDIA NIM 판단 완료 | 전체 {len(enriched)}건, "
        f"성공 {llm_success_count}건, 시도 {llm_attempt_count}건, quota_exhausted={quota_exhausted}, "
        f"provider={provider_counts}, relatedness={relatedness_counts}"
    )
    return state


print("NVIDIA NIM LLMJudgementAgent 정의 완료")


NVIDIA NIM LLMJudgementAgent 정의 완료


## Cell 16. Agent Node 7: CrossSourceCheckAgent

목적:
- 관련/부분 관련/관련 후보 기사 기준으로 위험 이벤트가 복수 언론사에서 확인되는지 집계한다.

입력값:
- `state["llm_enriched"]`

출력값:
- `state["cross_check_results"]`

다음 셀과의 연결 관계:
- Cell 17 SQLiteStorageAgent와 Cell 19 ExcelReflectionAgent가 교차확인 결과를 저장/검토한다.

사용자가 수정해도 되는 부분:
- 교차확인 기준 언론사 수
- evidence_summary 개수

주의사항:
- `outlet_count >= 2`이면 교차확인됨으로 표시한다.


In [15]:
# =========================================
# Cell 16. Agent Node 7: CrossSourceCheckAgent
# =========================================

def cross_source_check_agent_node(state: NewsNIMAgentState) -> NewsNIMAgentState:
    rows = []

    for article in state["llm_enriched"]:
        judgement = article.get("llm_judgement", {})
        relatedness = judgement.get("relatedness", "")

        if relatedness not in ["관련", "부분 관련", "관련 후보"]:
            continue
        if article.get("article_page_result") not in ["실제 기사", "본문 확인 제한"]:
            continue

        event_list = judgement.get("risk_events", [])
        if isinstance(event_list, str):
            event_list = [event_list]

        rule_events = [e.get("event_type", "") for e in article.get("rule_risk_events", []) if e.get("event_type")]
        merged_events = list(set(event_list + rule_events))

        for event_type in merged_events:
            if not event_type:
                continue
            rows.append({
                "event_type": clean_text(event_type),
                "outlet_name": article.get("outlet_name"),
                "article_id": article.get("article_id"),
                "title": clean_text(article.get("title"))
            })

    df = pd.DataFrame(rows)
    results = []

    if not df.empty:
        for event_type, g in df.groupby("event_type"):
            outlet_count = g["outlet_name"].nunique()
            article_count = g["article_id"].nunique()
            cross_source_confirmed = int(outlet_count >= 2)
            evidence_summary = " | ".join([f"{r['outlet_name']}: {clean_text(r['title'])[:80]}" for _, r in g.head(5).iterrows()])

            results.append({
                "cross_id": make_id(event_type, prefix="cross"),
                "event_type": clean_text(event_type),
                "outlet_count": int(outlet_count),
                "article_count": int(article_count),
                "cross_source_confirmed": cross_source_confirmed,
                "evidence_summary": clean_text(evidence_summary)
            })

    state["cross_check_results"] = results
    state["logs"].append(f"[CrossSourceCheckAgent] 위험 이벤트 {len(results)}개 교차확인 완료")
    return state


print("CrossSourceCheckAgent 정의 완료")


CrossSourceCheckAgent 정의 완료


## Cell 17. Agent Node 8: SQLiteStorageAgent

목적:
- 검색 작업, 후보 기사, 접근성, 메타데이터, 규칙 이벤트, LLM 판단, 교차확인 결과를 SQLite에 저장한다.

입력값:
- 전체 state 결과

출력값:
- SQLite DB
- `state["saved_article_count"]`

다음 셀과의 연결 관계:
- Cell 23에서 SQLite 결과를 DataFrame으로 확인한다.

사용자가 수정해도 되는 부분:
- 저장 컬럼
- INSERT 방식을 `INSERT OR REPLACE`에서 다른 방식으로 변경

주의사항:
- 저장 전 모든 텍스트에 `clean_text`를 적용한다.
- HTML 전문은 SQLite에 저장하지 않는다.


In [16]:
# =========================================
# Cell 17. Agent Node 8: SQLiteStorageAgent
# =========================================

def sqlite_storage_agent_node(state: NewsNIMAgentState) -> NewsNIMAgentState:
    create_sqlite_schema(state["sqlite_path"])
    conn = sqlite3.connect(state["sqlite_path"])
    cur = conn.cursor()

    for t in state["search_tasks"]:
        cur.execute("""
        INSERT OR REPLACE INTO search_tasks VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            t["task_id"], t["target_date"], clean_text(t["outlet_name"]), t["site_domain"],
            clean_text(t["keyword"]), clean_text(t["query"]), t["rss_url"], t["created_at"]
        ))

    for c in state["candidates"]:
        cur.execute("""
        INSERT OR REPLACE INTO article_candidates VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            c["candidate_id"], c["task_id"], c["target_date"], clean_text(c["outlet_name"]),
            c["site_domain"], clean_text(c["keyword"]), clean_text(c["candidate_title"]),
            c["candidate_url"], c["rss_link"], clean_text(c["rss_published"]),
            clean_text(c["rss_summary"]), c["discovered_at"]
        ))

    for a in state["access_checked"]:
        cur.execute("""
        INSERT OR REPLACE INTO article_access_checks VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            a["check_id"], a["candidate_id"], clean_text(a["outlet_name"]), a["candidate_url"],
            a.get("http_status"), a.get("final_url"), a.get("final_domain"),
            a.get("domain_matched"), clean_text(a.get("content_type")),
            clean_text(a.get("access_result")), clean_text(a.get("failure_reason")),
            clean_text(a.get("resolver_status")), clean_text(a.get("resolver_message")),
            a.get("checked_at")
        ))

    for m in state["article_verified"]:
        cur.execute("""
        INSERT OR REPLACE INTO article_metadata VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            m["article_id"], m["candidate_id"], clean_text(m["outlet_name"]),
            clean_text(m.get("title")), clean_text(m.get("author")),
            clean_text(m.get("published_at")), clean_text(m.get("published_date")),
            clean_text(m.get("description")), clean_text(m.get("canonical_url")),
            m.get("body_exists"), m.get("body_length"), clean_text(m.get("body_preview")),
            m.get("date_matched"), clean_text(m.get("article_page_result")),
            clean_text(m.get("metadata_quality")), m.get("verification_score"),
            clean_text(m.get("quality_grade"))
        ))

    for item in state["rule_enriched"]:
        for e in item.get("rule_risk_events", []):
            cur.execute("""
            INSERT OR REPLACE INTO rule_risk_events VALUES (?, ?, ?, ?, ?, ?, ?)
            """, (
                e["event_id"], e["article_id"], clean_text(e["outlet_name"]),
                clean_text(e["event_type"]), clean_text(e["event_keyword"]),
                clean_text(e["evidence_text"]), clean_text(e["relevance_result"])
            ))

    for item in state["llm_enriched"]:
        judgement = item.get("llm_judgement", {})
        cur.execute("""
        INSERT OR REPLACE INTO llm_judgements VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            make_id(item.get("article_id", "") + "_llm", prefix="judge"),
            item.get("article_id"),
            clean_text(item.get("outlet_name")),
            clean_text(judgement.get("relatedness")),
            clean_text(judgement.get("main_risk_event")),
            clean_text(json.dumps(judgement.get("risk_events", []), ensure_ascii=False)),
            clean_text(judgement.get("fact_opinion_forecast")),
            clean_text(judgement.get("event_summary")),
            clean_text(judgement.get("reason")),
            state.get("nvidia_nim_model"),
            clean_text(item.get("llm_provider_used")),
            item.get("llm_used", 0),
            datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        ))

    for c in state["cross_check_results"]:
        cur.execute("""
        INSERT OR REPLACE INTO cross_source_checks VALUES (?, ?, ?, ?, ?, ?)
        """, (
            c["cross_id"], clean_text(c["event_type"]), c["outlet_count"], c["article_count"],
            c["cross_source_confirmed"], clean_text(c["evidence_summary"])
        ))

    conn.commit()
    conn.close()

    state["saved_article_count"] = len(state["article_verified"])
    state["logs"].append(f"[SQLiteStorageAgent] SQLite 저장 완료: {state['sqlite_path']}")
    return state


print("SQLiteStorageAgent 정의 완료")


SQLiteStorageAgent 정의 완료


## Cell 18. Agent Node 9: ChromaStorageAgent

목적:
- 검증된 기사 중 RAG 활용 가능한 항목만 Chroma Vector DB에 저장한다.

입력값:
- `state["llm_enriched"]`
- `state["chroma_path"]`

출력값:
- Chroma DB
- `state["saved_vector_count"]`

다음 셀과의 연결 관계:
- Cell 24에서 Chroma RAG 검색을 테스트한다.

사용자가 수정해도 되는 부분:
- 저장 조건
- embedding model
- 저장 document 형식

주의사항:
- 기사 전문 전체를 저장하지 않는다.
- 실행별 신규 `chroma_path`를 사용해 readonly database 오류를 피한다.
- 저장 조건은 접근 가능, 도메인 일치, 실제 기사, 본문 존재, 원문 URL 존재, Google News URL 아님, 관련/부분 관련/관련 후보 중 하나다.


In [17]:
# =========================================
# Cell 18. Agent Node 9: ChromaStorageAgent
# =========================================

def is_valid_for_vector(article: dict, require_date_match: bool = False) -> bool:
    final_url = article.get("final_url", "")
    judgement = article.get("llm_judgement", {})
    relatedness = judgement.get("relatedness", "")

    if require_date_match and article.get("date_matched") != 1:
        return False

    return (
        article.get("access_result") == "접근 가능"
        and article.get("domain_matched") == 1
        and article.get("article_page_result") == "실제 기사"
        and article.get("body_exists") == 1
        and bool(final_url)
        and "news.google.com" not in extract_domain(final_url)
        and relatedness in ["관련", "부분 관련", "관련 후보"]
    )


def chroma_storage_agent_node(state: NewsNIMAgentState) -> NewsNIMAgentState:
    os.makedirs(state["chroma_path"], exist_ok=True)

    embedding_func = embedding_functions.SentenceTransformerEmbeddingFunction(
        model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )

    client = chromadb.PersistentClient(path=state["chroma_path"])
    collection = client.get_or_create_collection(
        name="ews_news_articles",
        embedding_function=embedding_func
    )

    ids, documents, metadatas = [], [], []

    for article in state["llm_enriched"]:
        if not is_valid_for_vector(article, state.get("require_date_match_for_vector", False)):
            continue

        judgement = article.get("llm_judgement", {})
        rule_events = [e.get("event_type", "") for e in article.get("rule_risk_events", []) if e.get("event_type")]
        risk_event_text = ", ".join(sorted(set(rule_events + judgement.get("risk_events", []))))

        # RAG에는 기사 전문이 아니라 검색과 검토에 필요한 요약 정보만 넣는다.
        doc = clean_text(f"""
제목: {article.get("title")}
언론사: {article.get("outlet_name")}
작성일자: {article.get("published_date")}
작성자: {article.get("author")}
관련성: {judgement.get("relatedness")}
주요 위험 이벤트: {judgement.get("main_risk_event")}
위험 이벤트 목록: {risk_event_text}
요약: {judgement.get("event_summary")}
판단 근거: {judgement.get("reason")}
본문 일부: {article.get("body_preview")}
URL: {article.get("final_url")}
""")

        ids.append(article["article_id"])
        documents.append(doc)
        metadatas.append({
            "article_id": article.get("article_id", ""),
            "outlet_name": clean_text(article.get("outlet_name", "")),
            "source_url": clean_text(article.get("final_url", "")),
            "published_date": clean_text(article.get("published_date", "")),
            "author": clean_text(article.get("author", "")),
            "article_page_result": clean_text(article.get("article_page_result", "")),
            "body_exists": int(article.get("body_exists", 0)),
            "quality_grade": clean_text(article.get("quality_grade", "")),
            "relatedness": clean_text(judgement.get("relatedness", "")),
            "risk_event_text": clean_text(risk_event_text),
            "llm_provider_used": clean_text(article.get("llm_provider_used", ""))
        })

    if ids:
        collection.upsert(ids=ids, documents=documents, metadatas=metadatas)

    state["saved_vector_count"] = len(ids)
    state["logs"].append(f"[ChromaStorageAgent] Chroma 저장 완료: {len(ids)}건 | {state['chroma_path']}")
    return state


print("ChromaStorageAgent 정의 완료")


ChromaStorageAgent 정의 완료


## Cell 19. Agent Node 10: ExcelReflectionAgent

목적:
- 수집가능성 검토표에 반영할 평가 결과와 근거를 생성하고, Excel 저장 helper를 정의한다.

입력값:
- 전체 state 결과

출력값:
- `state["excel_reflection"]`
- `save_excel_outputs(final_state)`

다음 셀과의 연결 관계:
- Cell 20 Workflow에 ExcelReflectionAgent가 연결되고, Cell 21 또는 Cell 25에서 Excel 파일을 저장한다.

사용자가 수정해도 되는 부분:
- 검토 항목
- 판단 기준
- Excel sheet 구성

주의사항:
- run 함수에서 이미 Excel을 저장했더라도 Cell 25에서 다시 저장해도 덮어쓰기만 수행하므로 문제 없다.


In [18]:
# =========================================
# Cell 19. Agent Node 10: ExcelReflectionAgent
# =========================================

def excel_reflection_agent_node(state: NewsNIMAgentState) -> NewsNIMAgentState:
    candidate_count = len(state.get("candidates", []))
    access_count = sum(1 for x in state.get("access_checked", []) if x.get("access_result") == "접근 가능")
    actual_article_count = sum(1 for x in state.get("article_verified", []) if x.get("article_page_result") == "실제 기사")
    body_count = sum(1 for x in state.get("article_verified", []) if x.get("body_exists") == 1)
    llm_success_count = sum(1 for x in state.get("llm_enriched", []) if x.get("llm_provider_used") == "nvidia_nim")
    related_count = sum(1 for x in state.get("llm_enriched", []) if x.get("llm_judgement", {}).get("relatedness") in ["관련", "부분 관련", "관련 후보"])
    cross_confirmed_count = sum(1 for x in state.get("cross_check_results", []) if x.get("cross_source_confirmed") == 1)
    vector_count = state.get("saved_vector_count", 0)

    def result(ok, mid=False):
        return "양호" if ok else ("부분 가능" if mid else "제한")

    reflection = {
        "원문 확보 가능성": {
            "검토 결과": result(access_count > 0, candidate_count > 0),
            "검토 결과 근거": f"후보 {candidate_count}건 중 접근 가능 URL {access_count}건"
        },
        "출처 URL 확보 가능성": {
            "검토 결과": result(access_count > 0),
            "검토 결과 근거": "Google News 중간 URL을 googlenewsdecoder로 원문 URL 해석 후 final_url을 저장"
        },
        "메타데이터 확보 가능성": {
            "검토 결과": result(actual_article_count > 0, access_count > 0),
            "검토 결과 근거": f"실제 기사 {actual_article_count}건, 본문 확인 {body_count}건"
        },
        "자동수집 가능성": {
            "검토 결과": result(candidate_count > 0 and access_count > 0),
            "검토 결과 근거": "RSS 수집, URL 해석, 접근성 검증, 본문 검증, 저장까지 Agent workflow로 자동 수행"
        },
        "수집 안정성": {
            "검토 결과": result(access_count >= max(1, candidate_count * 0.5), access_count > 0),
            "검토 결과 근거": f"후보 대비 접근 가능 비율: {access_count}/{candidate_count}"
        },
        "저작권·이용권한 검토": {
            "검토 결과": "추가 검토 필요",
            "검토 결과 근거": "기사 전문 전체를 저장하지 않고 URL, 메타데이터, 요약, 위험 이벤트, 판단 근거 중심으로 저장하도록 제한"
        },
        "RAG 활용 가능성": {
            "검토 결과": result(vector_count > 0, related_count > 0),
            "검토 결과 근거": f"Chroma 저장 {vector_count}건. 기사 전문이 아닌 검증 요약 문서를 저장"
        },
        "위험 이벤트 추출 가능성": {
            "검토 결과": result(related_count > 0),
            "검토 결과 근거": f"NVIDIA NIM 성공 판단 {llm_success_count}건, 관련/부분 관련/관련 후보 {related_count}건"
        },
        "교차확인 가능성": {
            "검토 결과": result(cross_confirmed_count > 0, len(state.get("cross_check_results", [])) > 0),
            "검토 결과 근거": f"위험 이벤트 교차확인 결과 {len(state.get('cross_check_results', []))}개, 복수 언론사 확인 {cross_confirmed_count}개"
        }
    }

    state["excel_reflection"] = reflection
    state["logs"].append("[ExcelReflectionAgent] 수집가능성 검토표 반영값 생성 완료")
    return state


def load_sqlite_result_frames(sqlite_path: str) -> dict:
    frames = {}
    if not sqlite_path or not os.path.exists(sqlite_path):
        return frames

    conn = sqlite3.connect(sqlite_path)
    table_names = [
        "search_tasks", "article_candidates", "article_access_checks", "article_metadata",
        "rule_risk_events", "llm_judgements", "cross_source_checks"
    ]
    for table in table_names:
        frames[table] = pd.read_sql_query(f"SELECT * FROM {table}", conn)
    conn.close()
    return frames


def save_excel_outputs(final_state: dict) -> str:
    frames = load_sqlite_result_frames(final_state["sqlite_path"])
    excel_path = final_state["excel_path"]
    os.makedirs(os.path.dirname(excel_path), exist_ok=True)

    with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
        frames.get("search_tasks", pd.DataFrame()).to_excel(writer, sheet_name="01_search_tasks", index=False)
        frames.get("article_candidates", pd.DataFrame()).to_excel(writer, sheet_name="02_candidates", index=False)
        frames.get("article_access_checks", pd.DataFrame()).to_excel(writer, sheet_name="03_access_checks", index=False)
        frames.get("article_metadata", pd.DataFrame()).to_excel(writer, sheet_name="04_article_metadata", index=False)
        frames.get("rule_risk_events", pd.DataFrame()).to_excel(writer, sheet_name="05_rule_events", index=False)
        frames.get("llm_judgements", pd.DataFrame()).to_excel(writer, sheet_name="06_llm_judgements", index=False)
        frames.get("cross_source_checks", pd.DataFrame()).to_excel(writer, sheet_name="07_cross_source", index=False)

        reflection_rows = []
        for item, value in final_state.get("excel_reflection", {}).items():
            reflection_rows.append({
                "검토 항목": clean_text(item),
                "검토 결과": clean_text(value.get("검토 결과", "")),
                "검토 결과 근거": clean_text(value.get("검토 결과 근거", ""))
            })
        pd.DataFrame(reflection_rows).to_excel(writer, sheet_name="08_excel_reflection", index=False)

        diagnosis_rows = [
            {"항목": "user_news_query", "값": final_state.get("user_news_query")},
            {"항목": "target_date", "값": final_state.get("target_date")},
            {"항목": "parsed_outlets", "값": ", ".join(final_state.get("parsed_outlets", []))},
            {"항목": "keywords", "값": ", ".join(final_state.get("keywords", []))},
            {"항목": "candidate_count", "값": len(final_state.get("candidates", []))},
            {"항목": "access_checked_count", "값": len(final_state.get("access_checked", []))},
            {"항목": "article_verified_count", "값": len(final_state.get("article_verified", []))},
            {"항목": "llm_enriched_count", "값": len(final_state.get("llm_enriched", []))},
            {"항목": "saved_vector_count", "값": final_state.get("saved_vector_count")},
            {"항목": "sqlite_path", "값": final_state.get("sqlite_path")},
            {"항목": "chroma_path", "값": final_state.get("chroma_path")},
            {"항목": "excel_path", "값": final_state.get("excel_path")},
            {"항목": "llm_provider", "값": final_state.get("llm_provider")},
            {"항목": "nvidia_nim_model", "값": final_state.get("nvidia_nim_model")},
            {"항목": "errors", "값": " | ".join(final_state.get("errors", []))}
        ]
        pd.DataFrame(diagnosis_rows).to_excel(writer, sheet_name="09_pipeline_diagnosis", index=False)

    return excel_path


print("ExcelReflectionAgent 및 Excel 저장 helper 정의 완료")


ExcelReflectionAgent 및 Excel 저장 helper 정의 완료


## Cell 20. Workflow Compile

목적:
- LangGraph workflow를 구성하고 Agent Node 실행 순서를 정의한다.

입력값:
- 각 Agent node 함수

출력값:
- `app`

다음 셀과의 연결 관계:
- Cell 21과 Cell 22에서 `app.invoke(initial_state)`로 전체 파이프라인을 실행한다.

사용자가 수정해도 되는 부분:
- Agent 순서
- 중간 조건 분기

주의사항:
- 현재 순서는 요청한 Agent 흐름을 따른다.


In [19]:
# =========================================
# Cell 20. Workflow Compile
# =========================================

workflow = StateGraph(NewsNIMAgentState)

workflow.add_node("query_planning", query_planning_agent_node)
workflow.add_node("rss_candidate", rss_candidate_agent_node)
workflow.add_node("url_access", url_access_agent_node)
workflow.add_node("article_verification", article_verification_agent_node)
workflow.add_node("rule_risk_event", rule_risk_event_agent_node)
workflow.add_node("llm_judgement", llm_judgement_agent_node)
workflow.add_node("cross_source_check", cross_source_check_agent_node)
workflow.add_node("sqlite_storage", sqlite_storage_agent_node)
workflow.add_node("chroma_storage", chroma_storage_agent_node)
workflow.add_node("excel_reflection", excel_reflection_agent_node)

workflow.set_entry_point("query_planning")
workflow.add_edge("query_planning", "rss_candidate")
workflow.add_edge("rss_candidate", "url_access")
workflow.add_edge("url_access", "article_verification")
workflow.add_edge("article_verification", "rule_risk_event")
workflow.add_edge("rule_risk_event", "llm_judgement")
workflow.add_edge("llm_judgement", "cross_source_check")
workflow.add_edge("cross_source_check", "sqlite_storage")
workflow.add_edge("sqlite_storage", "chroma_storage")
workflow.add_edge("chroma_storage", "excel_reflection")
workflow.add_edge("excel_reflection", END)

app = workflow.compile()

print("Workflow compile 완료")


Workflow compile 완료


## Cell 21. CONFIG 생성 함수와 전체 실행 함수

목적:
- 자연어 질의 파싱 결과를 받아 실행별 CONFIG를 생성하고, 전체 Agent 파이프라인을 한 번에 실행하는 함수를 정의한다.

입력값:
- `parsed_query_config`
- `user_query`

출력값:
- CONFIG dict
- final_state
- 자연어 응답

다음 셀과의 연결 관계:
- Cell 22 단건 실행, Cell 27 시뮬레이션 실행에서 사용한다.

사용자가 수정해도 되는 부분:
- 후보 수
- LLM 호출 수
- sleep 설정
- 저장 경로 prefix

주의사항:
- CONFIG를 전역 하드코딩하지 않고 자연어 질의 파싱 결과로 생성한다.
- run_id를 사용해 매 실행마다 SQLite, Chroma, Excel 경로를 새로 만든다.


In [20]:
# =========================================
# Cell 21. CONFIG 생성 함수와 전체 실행 함수
# =========================================

def build_config_from_user_query(parsed_query_config: dict) -> dict:
    run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
    target_date = parsed_query_config["target_date"]

    return {
        "target_date": target_date,
        "outlets": parsed_query_config["outlets"],
        "keywords": parsed_query_config["keywords"],
        "hl": "ko",
        "gl": "KR",
        "ceid": "KR:ko",
        "sleep_sec": 0.7,
        "max_candidates_per_task": 5,
        "max_total_candidates": 60,
        "llm_provider": "nvidia_nim",
        "nvidia_nim_base_url": "https://integrate.api.nvidia.com/v1",
        "nvidia_nim_model": "meta/llama-3.1-70b-instruct",
        "llm_sleep_sec": 3.0,
        "max_llm_calls": 15,
        "stop_llm_on_quota_error": True,
        "require_date_match_for_vector": False,
        "sqlite_path": f"/content/ews_news_nim_agent_{target_date.replace('-', '')}_run_{run_id}.db",
        "chroma_path": f"/content/ews_news_nim_chroma_{target_date.replace('-', '')}_run_{run_id}",
        "excel_path": f"/content/ews_news_nim_agent_{target_date.replace('-', '')}_run_{run_id}_result.xlsx",
        "user_news_query": parsed_query_config["original_query"],
        "parsed_outlets": parsed_query_config["selected_outlets"]
    }


def build_natural_language_response(final_state: dict, top_n: int = 10) -> str:
    user_query = final_state.get("user_news_query", "")
    target_date = final_state.get("target_date", "")
    parsed_outlets = final_state.get("parsed_outlets", [])
    keywords = final_state.get("keywords", [])
    articles = final_state.get("llm_enriched", [])

    selected = []
    for article in articles:
        judgement = article.get("llm_judgement", {})
        relatedness = judgement.get("relatedness", "")
        if relatedness not in ["관련", "부분 관련", "관련 후보"]:
            continue
        if article.get("article_page_result") not in ["실제 기사", "본문 확인 제한"]:
            continue
        selected.append(article)

    relatedness_rank = {"관련": 3, "부분 관련": 2, "관련 후보": 1}

    def sort_key(article):
        judgement = article.get("llm_judgement", {})
        provider = article.get("llm_provider_used", "")
        relatedness = judgement.get("relatedness", "")
        return (
            1 if provider == "nvidia_nim" else 0,
            relatedness_rank.get(relatedness, 0),
            1 if article.get("article_page_result") == "실제 기사" else 0,
            article.get("body_exists", 0),
            article.get("verification_score", 0)
        )

    selected = sorted(selected, key=sort_key, reverse=True)
    nim_success_count = sum(1 for x in articles if x.get("llm_provider_used") == "nvidia_nim")

    lines = []
    lines.append("========== 자연어 질의 응답 ==========")
    lines.append(f"사용자 질의: {user_query}")
    lines.append(f"기준일자: {target_date}")
    lines.append(f"대상 언론사: {', '.join(parsed_outlets)}")
    lines.append(f"검색 키워드: {', '.join(keywords)}")
    lines.append("")
    lines.append(f"수집 후보 수: {len(final_state.get('candidates', []))}")
    lines.append(f"접근 가능 URL 수: {sum(1 for x in final_state.get('access_checked', []) if x.get('access_result') == '접근 가능')}")
    lines.append(f"실제 기사 수: {sum(1 for x in final_state.get('article_verified', []) if x.get('article_page_result') == '실제 기사')}")
    lines.append(f"본문 확인 기사 수: {sum(1 for x in final_state.get('article_verified', []) if x.get('body_exists') == 1)}")
    lines.append(f"NVIDIA NIM 판단 성공 수: {nim_success_count}")
    lines.append(f"Chroma 저장 수: {final_state.get('saved_vector_count', 0)}")
    lines.append("")

    if not selected:
        lines.append("관련 기사 후보를 찾지 못했습니다.")
        return "\n".join(lines)

    lines.append(f"상위 관련 기사 목록 Top {min(top_n, len(selected))}:")
    lines.append("")

    for i, article in enumerate(selected[:top_n], start=1):
        judgement = article.get("llm_judgement", {})
        title = clean_text(article.get("title") or article.get("candidate_title", ""))
        outlet = clean_text(article.get("outlet_name", ""))
        published_date = clean_text(article.get("published_date", "")) or "작성일자 미확인"
        author = clean_text(article.get("author", "")) or "작성자 미확인"
        url = clean_text(article.get("final_url", ""))
        relatedness = clean_text(judgement.get("relatedness", ""))
        main_event = clean_text(judgement.get("main_risk_event", ""))
        summary = clean_text(judgement.get("event_summary", ""))
        provider = clean_text(article.get("llm_provider_used", ""))

        lines.append(f"{i}. [{relatedness}] {title}")
        lines.append(f"   - 언론사: {outlet}")
        lines.append(f"   - 작성일자: {published_date}")
        lines.append(f"   - 작성자: {author}")
        lines.append(f"   - 주요 위험 이벤트: {main_event if main_event else '미분류'}")
        lines.append(f"   - 요약: {summary if summary else '요약 없음'}")
        lines.append(f"   - 판단 방식: {provider}")
        lines.append(f"   - URL: {url}")
        lines.append("")

    lines.append("해석:")
    lines.append("- 이 결과는 Google News RSS 후보 수집, 원문 URL 해석, 기사 접근성 검증, 본문/메타데이터 검증, NVIDIA NIM 또는 규칙 기반 fallback 판단을 거쳐 생성되었습니다.")
    lines.append("- 기사 전문 전체는 저장하지 않고 URL, 메타데이터, 요약, 위험 이벤트, 판단 근거 중심으로 저장했습니다.")
    return "\n".join(lines)


def run_news_agent_from_query(user_query: str, run_excel: bool = True, top_n: int = 10):
    parsed = parse_user_news_query(user_query, use_nim_parser=True)
    config = build_config_from_user_query(parsed)
    initial_state = init_state(config)
    final_state = app.invoke(initial_state)

    final_state["sqlite_frames"] = load_sqlite_result_frames(final_state["sqlite_path"])
    response = build_natural_language_response(final_state, top_n=top_n)
    print(response)

    if run_excel:
        excel_path = save_excel_outputs(final_state)
        print("Excel 저장 완료:", excel_path)

    return final_state


print("CONFIG 생성 함수와 전체 실행 함수 정의 완료")


CONFIG 생성 함수와 전체 실행 함수 정의 완료


## Cell 22. 사용자 자연어 질의 입력 및 단건 실행

목적:
- 사용자가 직접 자연어 질의를 입력하고 전체 Agent 파이프라인을 실행한다.

입력값:
- `USER_NEWS_QUERY`

출력값:
- `USER_NEWS_QUERY_CONFIG`
- `CONFIG`
- `initial_state`
- `final_state`
- `response`

다음 셀과의 연결 관계:
- Cell 23, 24, 25, 26에서 `final_state`를 사용해 결과를 확인한다.

사용자가 수정해도 되는 부분:
- 입력 질의
- `top_n`

주의사항:
- 이 셀은 실제 RSS, URL 접근, NIM 호출, 저장을 수행하므로 시간이 걸릴 수 있다.


In [21]:
# =========================================
# Cell 22. 사용자 자연어 질의 입력 및 단건 실행
# =========================================

USER_NEWS_QUERY = input("뉴스 수집 질의를 입력하세요: ").strip()
if not USER_NEWS_QUERY:
    USER_NEWS_QUERY = "오늘일자 기준으로 조중동의 건설경기 관련 뉴스 기사를 탐색해줘"

USER_NEWS_QUERY_CONFIG = parse_user_news_query(USER_NEWS_QUERY, use_nim_parser=True)
CONFIG = build_config_from_user_query(USER_NEWS_QUERY_CONFIG)
initial_state = init_state(CONFIG)
final_state = app.invoke(initial_state)
final_state["sqlite_frames"] = load_sqlite_result_frames(final_state["sqlite_path"])
response = build_natural_language_response(final_state, top_n=10)
print(response)

excel_path = save_excel_outputs(final_state)
print("Excel 저장 완료:", excel_path)


뉴스 수집 질의를 입력하세요: 


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

========== 자연어 질의 응답 ==========
사용자 질의: 오늘일자 기준으로 조중동의 건설경기 관련 뉴스 기사를 탐색해줘
기준일자: 2026-05-15
대상 언론사: 조선일보, 중앙일보, 동아일보
검색 키워드: 건설경기, 건설수주, 미분양, PF 부실, 공사비 상승, 건설투자, 주택시장

수집 후보 수: 60
접근 가능 URL 수: 60
실제 기사 수: 23
본문 확인 기사 수: 23
NVIDIA NIM 판단 성공 수: 15
Chroma 저장 수: 6

상위 관련 기사 목록 Top 10:

1. [관련] 국토부, 주택건설업계와 '타운홀 미팅'… 김윤덕 장관
   - 언론사: 조선일보
   - 작성일자: 작성일자 미확인
   - 작성자: 조선비즈
   - 주요 위험 이벤트: 공사비 상승
   - 요약: 국토부는 주택건설업계와 타운홀 미팅을 개최하여 주택 공급을 정상화하기 위한 해법을 모색하고, 공사비 상승 및 자재 수급 문제 등 애로사항을 논의했다.
   - 판단 방식: nvidia_nim
   - URL: https://biz.chosun.com/real_estate/real_estate_general/2026/05/14/W5FLP2SEUVDSRE4E7RSX6E5MMU/?outputType=amp

2. [관련] 경제부총리
   - 언론사: 조선일보
   - 작성일자: 2026-05-14
   - 작성자: 조선비즈
   - 주요 위험 이벤트: 착공 지연
   - 요약: 경제부총리가 태릉 골프장 부지에 주택 공급을 1년 앞당겨 2029년 착공하기로 결정
   - 판단 방식: nvidia_nim
   - URL: https://biz.chosun.com/policy/policy_sub/2026/05/15/32X5GGT75BF5JNJQ5NNY47RJIE/

3. [부분 관련] 금호건설, 1분기 영업익 121억… 전년比 111.2% 증가
   - 언론사: 조선일보
   - 작성일자: 2026-05-15
   - 작성자: 이성은 기자
   - 주요 위험 이

## Cell 23. SQLite 결과 확인

목적:
- SQLite에 저장된 결과를 DataFrame으로 로드해 확인한다.

입력값:
- `final_state["sqlite_path"]`

출력값:
- `df_tasks`
- `df_candidates`
- `df_access`
- `df_metadata`
- `df_rule_events`
- `df_llm`
- `df_cross`

다음 셀과의 연결 관계:
- Cell 25 Excel 저장 셀이 이 DataFrame 또는 SQLite를 기반으로 저장 결과를 재생성한다.

사용자가 수정해도 되는 부분:
- 표시할 컬럼
- `head()` 개수

주의사항:
- `final_state`가 없거나 SQLite 파일이 없으면 오류 대신 안내 메시지를 출력한다.


In [22]:
# =========================================
# Cell 23. SQLite 결과 확인
# =========================================

if "final_state" not in globals() or not final_state or not os.path.exists(final_state.get("sqlite_path", "")):
    print("아직 실행된 final_state 또는 SQLite 파일이 없습니다. Cell 22 또는 Cell 27에서 파이프라인을 먼저 실행하세요.")
else:
    frames = load_sqlite_result_frames(final_state["sqlite_path"])
    df_tasks = frames.get("search_tasks", pd.DataFrame())
    df_candidates = frames.get("article_candidates", pd.DataFrame())
    df_access = frames.get("article_access_checks", pd.DataFrame())
    df_metadata = frames.get("article_metadata", pd.DataFrame())
    df_rule_events = frames.get("rule_risk_events", pd.DataFrame())
    df_llm = frames.get("llm_judgements", pd.DataFrame())
    df_cross = frames.get("cross_source_checks", pd.DataFrame())

    print("search_tasks:", df_tasks.shape)
    print("article_candidates:", df_candidates.shape)
    print("article_access_checks:", df_access.shape)
    print("article_metadata:", df_metadata.shape)
    print("rule_risk_events:", df_rule_events.shape)
    print("llm_judgements:", df_llm.shape)
    print("cross_source_checks:", df_cross.shape)

    display(df_candidates.head(10))
    display(df_access.head(10))
    if not df_metadata.empty:
        display(df_metadata[[
            "article_id", "outlet_name", "title", "author",
            "published_date", "body_exists", "body_length",
            "date_matched", "article_page_result", "quality_grade"
        ]].head(15))
    display(df_rule_events.head(10))
    display(df_llm.head(15))
    display(df_cross.head(10))


search_tasks: (28, 8)
article_candidates: (60, 12)
article_access_checks: (60, 14)
article_metadata: (60, 17)
rule_risk_events: (22, 7)
llm_judgements: (60, 13)
cross_source_checks: (10, 6)


,candidate_id,task_id,target_date,outlet_name,site_domain,keyword,candidate_title,candidate_url,rss_link,rss_published,rss_summary,discovered_at
0,cand_5ae2f477dde4a17b75dba358602c0a5c,task_a4d8bcebed6eacc40507f70cf7cb54c5,2026-05-15,조선일보,chosun.com,건설경기,"이마트, 신세계건설에 5000억원 규모 유상증자… ""재무구조 개선"" - 조선비즈 -...",https://news.google.com/rss/articles/CBMiigFBV...,https://news.google.com/rss/articles/CBMiigFBV...,"Thu, 14 May 2026 08:47:00 GMT","<a href=""https://news.google.com/rss/articles/...",2026-05-15 05:16:52
1,cand_01552ffc9bc75bf54d5b39bf7227989e,task_a4d8bcebed6eacc40507f70cf7cb54c5,2026-05-15,조선일보,chosun.com,건설경기,"이마트, 신세계건설에 5000억원 규모 유상증자… ""재무구조 개선"" - 조선비즈 -...",https://news.google.com/rss/articles/CBMingFBV...,https://news.google.com/rss/articles/CBMingFBV...,"Thu, 14 May 2026 08:47:00 GMT","<a href=""https://news.google.com/rss/articles/...",2026-05-15 05:16:52
2,cand_79df81ceddf4d35584312c2ffd259854,task_a4d8bcebed6eacc40507f70cf7cb54c5,2026-05-15,조선일보,chosun.com,건설경기,"[동서남북] 자가 보유율 94% 루마니아, 왜 주거지옥일까 - 조선일보",https://news.google.com/rss/articles/CBMiiwFBV...,https://news.google.com/rss/articles/CBMiiwFBV...,"Thu, 14 May 2026 14:39:00 GMT","<a href=""https://news.google.com/rss/articles/...",2026-05-15 05:16:52
3,cand_95c7d165202b55d322a8d010822dbf2b,task_a4d8bcebed6eacc40507f70cf7cb54c5,2026-05-15,조선일보,chosun.com,건설경기,"최인호 HUG 사장, PF 사업장 현장 점검… ""맞춤형 금융 지원으로 건설업 활력 ...",https://news.google.com/rss/articles/CBMimAFBV...,https://news.google.com/rss/articles/CBMimAFBV...,"Thu, 14 May 2026 08:08:00 GMT","<a href=""https://news.google.com/rss/articles/...",2026-05-15 05:16:52
4,cand_828e1ca4233c9c3a9685629b513d33b6,task_a4d8bcebed6eacc40507f70cf7cb54c5,2026-05-15,조선일보,chosun.com,건설경기,"국토부, 주택건설업계와 '타운홀 미팅'… 김윤덕 장관 ""주택공급 현장서 해법 모색""...",https://news.google.com/rss/articles/CBMirAFBV...,https://news.google.com/rss/articles/CBMirAFBV...,"Thu, 14 May 2026 07:00:00 GMT","<a href=""https://news.google.com/rss/articles/...",2026-05-15 05:16:52
5,cand_209158396b87f00711f1187dc33f8682,task_35e6f3ae0caaa5a99ae62c48a26f72a6,2026-05-15,조선일보,chosun.com,건설수주,"금호건설, 1분기 영업익 121억… 전년比 111.2% 증가 - IT조선",https://news.google.com/rss/articles/CBMicEFVX...,https://news.google.com/rss/articles/CBMicEFVX...,"Fri, 15 May 2026 02:29:30 GMT","<a href=""https://news.google.com/rss/articles/...",2026-05-15 05:16:53
6,cand_d4f08b3f67e7510dd818d206fd8d9672,task_35e6f3ae0caaa5a99ae62c48a26f72a6,2026-05-15,조선일보,chosun.com,건설수주,원전 수출 '집안 싸움' 차단 - 조선일보,https://news.google.com/rss/articles/CBMijgFBV...,https://news.google.com/rss/articles/CBMijgFBV...,"Thu, 14 May 2026 15:32:00 GMT","<a href=""https://news.google.com/rss/articles/...",2026-05-15 05:16:53
7,cand_fe245e2fcba597e60d6b81d9be98006e,task_35e6f3ae0caaa5a99ae62c48a26f72a6,2026-05-15,조선일보,chosun.com,건설수주,"현대모비스, 반도체·로보틱스 미래 사업 박차… ""시장 주도할 기술 적시에 제안"" -...",https://news.google.com/rss/articles/CBMikwFBV...,https://news.google.com/rss/articles/CBMikwFBV...,"Thu, 14 May 2026 07:04:00 GMT","<a href=""https://news.google.com/rss/articles/...",2026-05-15 05:16:53
8,cand_02c5f1db99faa387466738bfd2ce320c,task_9385be5ccc3a4ddf0df53c150be92786,2026-05-15,조선일보,chosun.com,미분양,"""전세계약 전 상담 받으세요""… 국토부·HUG, 안전계약 컨설팅 사업 시행 - 조선...",https://news.google.com/rss/articles/CBMimAFBV...,https://news.google.com/rss/articles/CBMimAFBV...,"Thu, 14 May 2026 21:03:23 GMT","<a href=""https://news.google.com/rss/articles/...",2026-05-15 05:16:54
9,cand_21d95ca0bff08b5963e54cd9c1660d15,task_9385be5ccc3a4ddf0df53c150be92786,2026-05-15,조선일보,chosun.com,미분양,"李 대통령, 성남 모란시장 방문…""대추 좀 사시라"" 홍보 - Chosunbiz",https://news.google.com/rss/articles/CBMimgFBV...,https://news.google.com/rss/articles/CBMimgFBV...,"Thu, 14 May 2026 11:18:00 GMT","<a href=""https://news.google.com/rss/articles/...",2026-05-15 05:16:54


,check_id,candidate_id,outlet_name,candidate_url,http_status,final_url,final_domain,domain_matched,content_type,access_result,failure_reason,resolver_status,resolver_message,checked_at
0,chk_1f02ad09fcc6d51684d5efaa640ac785,cand_5ae2f477dde4a17b75dba358602c0a5c,조선일보,https://news.google.com/rss/articles/CBMiigFBV...,200,https://biz.chosun.com/distribution/channel/20...,biz.chosun.com,1,text/html; charset=utf-8,접근 가능,,resolved,Google News URL 원문 URL 해석 성공,2026-05-15 05:17:20
1,chk_b9e1b1b25e80833611baf6b2cfc471fb,cand_01552ffc9bc75bf54d5b39bf7227989e,조선일보,https://news.google.com/rss/articles/CBMingFBV...,200,https://biz.chosun.com/distribution/channel/20...,biz.chosun.com,1,text/html; charset=utf-8,접근 가능,,resolved,Google News URL 원문 URL 해석 성공,2026-05-15 05:17:22
2,chk_a6b8e435de1506d39baf200eaa46cc6e,cand_79df81ceddf4d35584312c2ffd259854,조선일보,https://news.google.com/rss/articles/CBMiiwFBV...,200,https://www.chosun.com/opinion/dongseonambuk/2...,chosun.com,1,text/html; charset=utf-8,접근 가능,,resolved,Google News URL 원문 URL 해석 성공,2026-05-15 05:17:27
3,chk_fd20bffacc4b6c7f0bf4c42060dc76f4,cand_95c7d165202b55d322a8d010822dbf2b,조선일보,https://news.google.com/rss/articles/CBMimAFBV...,200,https://biz.chosun.com/real_estate/real_estate...,biz.chosun.com,1,text/html; charset=utf-8,접근 가능,,resolved,Google News URL 원문 URL 해석 성공,2026-05-15 05:17:29
4,chk_d32526bdfcc9cc55ed1bc3497a528362,cand_828e1ca4233c9c3a9685629b513d33b6,조선일보,https://news.google.com/rss/articles/CBMirAFBV...,200,https://biz.chosun.com/real_estate/real_estate...,biz.chosun.com,1,text/html; charset=utf-8,접근 가능,,resolved,Google News URL 원문 URL 해석 성공,2026-05-15 05:17:33
5,chk_03a8844e49e438d05ddb4c080b9ad6e2,cand_209158396b87f00711f1187dc33f8682,조선일보,https://news.google.com/rss/articles/CBMicEFVX...,200,https://it.chosun.com/news/articleView.html?id...,it.chosun.com,1,text/html; charset=utf-8,접근 가능,,resolved,Google News URL 원문 URL 해석 성공,2026-05-15 05:17:37
6,chk_c47968ac72c3c24fd024c71562f8d08b,cand_d4f08b3f67e7510dd818d206fd8d9672,조선일보,https://news.google.com/rss/articles/CBMijgFBV...,200,https://www.chosun.com/economy/economy_general...,chosun.com,1,text/html; charset=utf-8,접근 가능,,resolved,Google News URL 원문 URL 해석 성공,2026-05-15 05:17:40
7,chk_996ed973a6e5ea6dd6fb8c6185bd4c8b,cand_fe245e2fcba597e60d6b81d9be98006e,조선일보,https://news.google.com/rss/articles/CBMikwFBV...,200,https://biz.chosun.com/industry/car/2026/05/14...,biz.chosun.com,1,text/html; charset=utf-8,접근 가능,,resolved,Google News URL 원문 URL 해석 성공,2026-05-15 05:17:42
8,chk_975e64f08d66df6dc49d5df35c9dc42d,cand_02c5f1db99faa387466738bfd2ce320c,조선일보,https://news.google.com/rss/articles/CBMimAFBV...,200,https://biz.chosun.com/real_estate/real_estate...,biz.chosun.com,1,text/html; charset=utf-8,접근 가능,,resolved,Google News URL 원문 URL 해석 성공,2026-05-15 05:17:47
9,chk_1c6e6e36bdf45d4ef6bc7e275dd81a2c,cand_21d95ca0bff08b5963e54cd9c1660d15,조선일보,https://news.google.com/rss/articles/CBMimgFBV...,200,https://biz.chosun.com/policy/politics/preside...,biz.chosun.com,1,text/html; charset=utf-8,접근 가능,,resolved,Google News URL 원문 URL 해석 성공,2026-05-15 05:17:49


,article_id,outlet_name,title,author,published_date,body_exists,body_length,date_matched,article_page_result,quality_grade
0,art_02de781936b73e6f055ea0e975eb5369,조선일보,"이마트, 신세계건설에 5000억원 규모 유상증자…",조선비즈,2026-05-14,0,0,0,본문 확인 제한,중간
1,art_3e335185eac605e00f82f58c9dd65cc4,조선일보,"이마트, 신세계건설에 5000억원 규모 유상증자…",조선비즈,,1,692,0,실제 기사,중간
2,art_2a32daf94b57a6468f6cf20280ae66a2,조선일보,"[동서남북] 자가 보유율 94% 루마니아, 왜 주거지옥일까",조선일보,2026-04-28,0,0,0,본문 확인 제한,중간
3,art_9e9b98d7aaca50dde04e21ec791684bb,조선일보,"최인호 HUG 사장, PF 사업장 현장 점검…",조선비즈,2026-05-14,0,0,0,본문 확인 제한,중간
4,art_f92a1b28b8732eb1adfccac3ab424592,조선일보,"국토부, 주택건설업계와 '타운홀 미팅'… 김윤덕 장관",조선비즈,,1,870,0,실제 기사,중간
5,art_265ad1794e0e2f4b14bed3323291dd09,조선일보,"금호건설, 1분기 영업익 121억… 전년比 111.2% 증가",이성은 기자,2026-05-15,1,599,1,실제 기사,높음
6,art_e7785c92d073d99c998d785b86098b66,조선일보,원전 수출 '집안 싸움' 차단,조선일보,2026-05-14,0,0,0,본문 확인 제한,중간
7,art_12d05f262a598daed6613d0b86208e9b,조선일보,"현대모비스, 반도체·로보틱스 미래 사업 박차…",조선비즈,,1,1233,0,실제 기사,중간
8,art_e5bd74678050f7233e554b6788260fdc,조선일보,"""전세계약 전 상담 받으세요""… 국토부·HUG, 안전계약 컨설팅 사업 시행 - 조선비즈",조선비즈,2026-05-14,0,0,0,본문 확인 제한,중간
9,art_8e16e082dedf67661e63356f4ead5b36,조선일보,"李 대통령, 성남 모란시장 방문…",조선비즈,2026-05-14,0,0,0,본문 확인 제한,중간


,event_id,article_id,outlet_name,event_type,event_keyword,evidence_text,relevance_result
0,rule_evt_cdf6fef2de9fc9772ea119bb9a8f306b,art_02de781936b73e6f055ea0e975eb5369,조선일보,기업 재무 건전성,재무,"이마트, 신세계건설에 5000억원 규모 유상증자… ""재무구조 개선"" - 조선비즈 -...",관련 후보
1,rule_evt_4252d55da0cf23bcc9f51612047fe9f7,art_3e335185eac605e00f82f58c9dd65cc4,조선일보,건설수주 감소,신규 수주,"이마트, 신세계건설에 5000억원 규모 유상증자… ""재무구조 개선"" - 조선비즈 -...",관련 후보
2,rule_evt_fede79315674de91ef5918a4ffc3db84,art_3e335185eac605e00f82f58c9dd65cc4,조선일보,기업 재무 건전성,재무,"이마트, 신세계건설에 5000억원 규모 유상증자… ""재무구조 개선"" - 조선비즈 -...",관련 후보
3,rule_evt_5d371cbfc36020a2f188b488ef79c70b,art_9e9b98d7aaca50dde04e21ec791684bb,조선일보,PF 부실,PF,"최인호 HUG 사장, PF 사업장 현장 점검… ""맞춤형 금융 지원으로 건설업 활력 ...",관련 후보
4,rule_evt_a1bab35c6f4986be3691cfadc5126e9c,art_f92a1b28b8732eb1adfccac3ab424592,조선일보,공사비 상승,공사비,"국토부, 주택건설업계와 '타운홀 미팅'… 김윤덕 장관 ""주택공급 현장서 해법 모색""...",관련 후보
5,rule_evt_3217ea2b1f7642dae95008b64e63b3ab,art_f92a1b28b8732eb1adfccac3ab424592,조선일보,착공 지연,인허가,"국토부, 주택건설업계와 '타운홀 미팅'… 김윤덕 장관 ""주택공급 현장서 해법 모색""...",관련 후보
6,rule_evt_19ea2af4487571347eee45c602ab6b32,art_f92a1b28b8732eb1adfccac3ab424592,조선일보,주택시장 위험,주택공급,"국토부, 주택건설업계와 '타운홀 미팅'… 김윤덕 장관 ""주택공급 현장서 해법 모색""...",관련 후보
7,rule_evt_0af610eb1ba535df2ae51c0de1d1918c,art_265ad1794e0e2f4b14bed3323291dd09,조선일보,기업 재무 건전성,유동성,"금호건설, 1분기 영업익 121억… 전년比 111.2% 증가 - IT조선 금호건설,...",관련 후보
8,rule_evt_931dc8253ad18808328afd7a7f137287,art_f54d368cca636f3b0ca66f4475259b6c,조선일보,공사비 상승,공사비,신혼희망타운 입주까지 9년 걸릴 판… 공공주택 공급 곳곳서 '멈춤' - 조선비즈 -...,관련 후보
9,rule_evt_2f729cc14fa9bb4198bb2367c5572b63,art_f54d368cca636f3b0ca66f4475259b6c,조선일보,주택시장 위험,분양,신혼희망타운 입주까지 9년 걸릴 판… 공공주택 공급 곳곳서 '멈춤' - 조선비즈 -...,관련 후보


,judgement_id,article_id,outlet_name,relatedness,main_risk_event,risk_events_json,fact_opinion_forecast,event_summary,judgement_reason,llm_model,llm_provider_used,llm_used,created_at
0,judge_7d5c181cfda6a30b2cdba9a3a4ae7cf5,art_02de781936b73e6f055ea0e975eb5369,조선일보,부분 관련,기업 재무 건전성,"[""기업 재무 건전성""]",사실,이마트가 신세계건설에 5000억원 규모의 유상증자를 통해 재무구조를 개선할 예정이다.,이 기사는 신세계건설의 재무구조 개선에 대한 내용을 다루고 있기 때문에 건설경기 간...,meta/llama-3.1-70b-instruct,nvidia_nim,1,2026-05-15 05:25:51
1,judge_edd8b905a748a5a81da6ac7d5167275c,art_3e335185eac605e00f82f58c9dd65cc4,조선일보,부분 관련,기업 재무 건전성,"[""건설수주 감소"", ""기업 재무 건전성""]",사실,이마트가 자회사 신세계건설에 5000억원 규모 유상증자를 통해 재무구조 개선과 운영...,"이 기사는 건설경기와 직접 관련된 건설수주나 미분양 등에 대한 언급은 없지만, 건설...",meta/llama-3.1-70b-instruct,nvidia_nim,1,2026-05-15 05:25:51
2,judge_ac8e6c99570f47613578ec7abdd88814,art_2a32daf94b57a6468f6cf20280ae66a2,조선일보,검증 제외,,[],,,URL/기사 검증 조건 또는 위험 신호 조건 미충족,meta/llama-3.1-70b-instruct,fallback,0,2026-05-15 05:25:51
3,judge_92da12c9741db91a6c4019903569325e,art_9e9b98d7aaca50dde04e21ec791684bb,조선일보,부분 관련,PF 부실,"[""PF 부실""]",사실,최인호 HUG 사장이 PF 사업장 현장 점검을 통해 건설업 활력 제고를 위한 맞춤형...,기사는 PF 사업장 현장 점검을 통해 건설업 활력 제고를 위한 맞춤형 금융 지원을 ...,meta/llama-3.1-70b-instruct,nvidia_nim,1,2026-05-15 05:25:51
4,judge_ea94422dff48c270caff7546f636477b,art_f92a1b28b8732eb1adfccac3ab424592,조선일보,관련,공사비 상승,"[""공사비 상승"", ""착공 지연"", ""주택시장 위험""]",사실,국토부는 주택건설업계와 타운홀 미팅을 개최하여 주택 공급을 정상화하기 위한 해법을 ...,"국토부의 주택건설업계와의 타운홀 미팅은 건설경기 관련 언론보도로 활용 가능하며, 공...",meta/llama-3.1-70b-instruct,nvidia_nim,1,2026-05-15 05:25:51
5,judge_be1b11c7709a6a45113ec53c3f2451a5,art_265ad1794e0e2f4b14bed3323291dd09,조선일보,부분 관련,기업 재무 건전성,"[""기업 재무 건전성""]",사실,금호건설은 1분기 영업이익 121억원을 기록해 전년 동기 대비 111.2% 증가한 ...,"금호건설의 재무 건전성 강화 기조와 실적 개선에 대한 언급이 있지만, 건설경기 전반...",meta/llama-3.1-70b-instruct,nvidia_nim,1,2026-05-15 05:25:51
6,judge_fe2c1098fd4f75592038c7a5116f066b,art_e7785c92d073d99c998d785b86098b66,조선일보,검증 제외,,[],,,URL/기사 검증 조건 또는 위험 신호 조건 미충족,meta/llama-3.1-70b-instruct,fallback,0,2026-05-15 05:25:51
7,judge_46d17c4ca9b7b450b31cb5d68789d32c,art_12d05f262a598daed6613d0b86208e9b,조선일보,검증 제외,,[],,,URL/기사 검증 조건 또는 위험 신호 조건 미충족,meta/llama-3.1-70b-instruct,fallback,0,2026-05-15 05:25:51
8,judge_473e1e6141f7777af49536fc3a4fe0bb,art_e5bd74678050f7233e554b6788260fdc,조선일보,검증 제외,,[],,,URL/기사 검증 조건 또는 위험 신호 조건 미충족,meta/llama-3.1-70b-instruct,fallback,0,2026-05-15 05:25:51
9,judge_35396e06bc15c5c2d6e3af8db3539c4d,art_8e16e082dedf67661e63356f4ead5b36,조선일보,검증 제외,,[],,,URL/기사 검증 조건 또는 위험 신호 조건 미충족,meta/llama-3.1-70b-instruct,fallback,0,2026-05-15 05:25:51


,cross_id,event_type,outlet_count,article_count,cross_source_confirmed,evidence_summary
0,cross_9c941988e30885d5db5a7b7b8d0eac99,PF 부실,2,2,1,"조선일보: 최인호 HUG 사장, PF 사업장 현장 점검… | 중앙일보: 김윤덕 국토..."
1,cross_7d5cfe9497a8626739375a37368993cc,건설경기 둔화,1,1,0,"조선일보: 국토부, 주택건설업계와 '타운홀 미팅'… 김윤덕 장관"
2,cross_f120c0b9cd68af1aaac7deafa35826c9,건설수주 감소,1,1,0,"조선일보: 이마트, 신세계건설에 5000억원 규모 유상증자…"
3,cross_277a3f251b83f5b92861f338a3879bf4,공급 속도전,1,1,0,"중앙일보: 서울 집값 들썩, 정부 공급 속도전…"
4,cross_fdcc1127127e58bbf1499f6409b9c264,공사비 상승,1,2,0,"조선일보: 국토부, 주택건설업계와 '타운홀 미팅'… 김윤덕 장관 | 조선일보: 신혼..."
5,cross_580e04a209ef73eb03edf8fc863ff50c,기업 재무 건전성,1,3,0,"조선일보: 이마트, 신세계건설에 5000억원 규모 유상증자… | 조선일보: 이마트,..."
6,cross_2b3c3776e9a1c64eb19ceb75b2430d86,주택공급 부족,1,1,0,"조선일보: 국토부, 주택건설업계와 '타운홀 미팅'… 김윤덕 장관"
7,cross_aa4aa30d3847a3a6f33c38420703e5af,주택시장 불안,1,1,0,"중앙일보: 서울 집값 들썩, 정부 공급 속도전…"
8,cross_b2d7a6aa26fbba8ba6f227eb2557ba68,주택시장 위험,3,8,1,"조선일보: 국토부, 주택건설업계와 '타운홀 미팅'… 김윤덕 장관 | 조선일보: 신혼..."
9,cross_81745bd8f720100f9bff22cad401de00,착공 지연,2,6,1,"조선일보: 국토부, 주택건설업계와 '타운홀 미팅'… 김윤덕 장관 | 조선일보: 정부..."


## Cell 24. Chroma RAG 검색 테스트

목적:
- Chroma에 저장된 검증 기사 요약 문서가 RAG 검색에 활용 가능한지 테스트한다.

입력값:
- `final_state["chroma_path"]`
- `RAG_QUERY`

출력값:
- Vector 검색 결과와 metadata

다음 셀과의 연결 관계:
- Cell 25 Excel 저장과 독립적으로 실행 가능하다.

사용자가 수정해도 되는 부분:
- `RAG_QUERY`
- `n_results`

주의사항:
- Chroma 저장 조건을 통과한 문서가 없으면 검색을 수행하지 않는다.


In [23]:
# =========================================
# Cell 24. Chroma RAG 검색 테스트
# =========================================

RAG_QUERY = "건설경기 위험 신호 미분양 PF 공사비 상승"

if "final_state" not in globals() or not final_state:
    print("아직 final_state가 없습니다. Cell 22 또는 Cell 27에서 파이프라인을 먼저 실행하세요.")
else:
    chroma_path_for_query = final_state.get("chroma_path", "")
    print("사용 Chroma 경로:", chroma_path_for_query)

    embedding_func = embedding_functions.SentenceTransformerEmbeddingFunction(
        model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )

    client = chromadb.PersistentClient(path=chroma_path_for_query)
    collection = client.get_or_create_collection(
        name="ews_news_articles",
        embedding_function=embedding_func
    )

    count = collection.count()
    print("Chroma 저장 문서 수:", count)

    if count == 0:
        print("검증된 기사 문서가 없어 Vector 검색을 수행하지 않습니다.")
    else:
        results = collection.query(
            query_texts=[RAG_QUERY],
            n_results=min(5, count)
        )

        print("========== Vector 검색 결과 ==========")
        for i, doc in enumerate(results["documents"][0]):
            print(f"\n--- 결과 {i+1} ---")
            print(doc)
            print("metadata:", results["metadatas"][0][i])


사용 Chroma 경로: /content/ews_news_nim_chroma_20260515_run_20260515_051652
Chroma 저장 문서 수: 6
========== Vector 검색 결과 ==========

--- 결과 1 ---
제목: 김윤덕 국토부 장관 주재 '주택건설업계 타운홀 미팅' 개최 | 중앙일보 언론사: 중앙일보 작성일자: 2026-05-15 작성자: https://www.joongang.co.kr 관련성: 부분 관련 주요 위험 이벤트: PF 부실 위험 이벤트 목록: PF 부실, 주택시장 위험, 착공 지연 요약: 국토부 장관이 주택건설업계와의 타운홀 미팅에서 PF 정상화, 정비사업 이주비 대출 규제 완화 등 19개 안건을 논의했다. 판단 근거: 기사 내용은 건설경기와 관련된 주택공급, PF 부실, 착공 지연 등에 대한 논의를 포함하고 있지만, 건설경기의 직접적인 신호는 아니다. 따라서 부분 관련으로 판단한다. 본문 일부: 김윤덕 국토교통부 장관이 14일 오전 강남 건설회관에서 주택건설업계 130여 명을 초청해 '주택건설업계 타운홀 미팅'을 개최했다. 이날 회의에는 정비사업·아파트·비(非)아파트(도시형생활주택·다세대·다가구)·건설임대·신축매입약정·신탁사 등 분야별 업계 관계자가 참석했으며, 국토부에서는 건설정책국장·주택정책관과 분야별 담당 과장 9인이 함께했다. 김 장관 "주택공급 외에는 다른 것 생각하지 않겠다" 김윤덕 장관은 모두발언에서 오월동주(吳越同舟)·와신상담(臥薪嘗膽) 고사를 인용하며 "현 국토부 장관은 주택공급 외에는 다른 것을 생각하지 않겠다"는 의지를 표명했다. 이어 "이번 타운홀은 일회성 행사가 아닌 정례 협의체로 운영하며, 6월 중하순 2차 회동을 갖겠다"며 ▶모든 건의를 넘버링·범주화 ▶국토부 단독 처리 사안과 타 부처(금융위·재경부 등) 협의 사안 분류 ▶업계 대표자 약 20명으로 운영위원회 구성 등 후속 운영방침을 제시했다. 회의에서는 PF(프로젝트 파이낸싱) 정상화, 정비사업 이주비 대출 규제, 신축매입약

## Cell 25. Excel 저장

목적:
- SQLite 결과와 Excel reflection 결과를 Excel workbook으로 저장한다.

입력값:
- `final_state`

출력값:
- `excel_path`

다음 셀과의 연결 관계:
- Cell 26 자연어 응답 출력과 독립적으로 실행 가능하다.

사용자가 수정해도 되는 부분:
- sheet 이름
- 저장 경로

주의사항:
- `run_news_agent_from_query` 또는 Cell 22에서 이미 저장했다면 이 셀을 다시 실행해도 같은 경로에 덮어쓴다.


In [24]:
# =========================================
# Cell 25. Excel 저장
# =========================================

if "final_state" not in globals() or not final_state:
    print("아직 final_state가 없습니다. Cell 22 또는 Cell 27에서 파이프라인을 먼저 실행하세요.")
else:
    excel_path = save_excel_outputs(final_state)
    print("Excel 저장 완료:", excel_path)


Excel 저장 완료: /content/ews_news_nim_agent_20260515_run_20260515_051652_result.xlsx


## Cell 26. 자연어 질의 응답 결과 출력

목적:
- 최종 사용자에게 보여줄 자연어 응답을 다시 생성해 출력한다.

입력값:
- `final_state`
- `top_n`

출력값:
- 자연어 응답 문자열

다음 셀과의 연결 관계:
- Cell 27 여러 질의 테스트 후에도 최신 `final_state` 기준으로 재출력할 수 있다.

사용자가 수정해도 되는 부분:
- `top_n`

주의사항:
- 최신 실행 결과를 기준으로 출력한다.


In [25]:
# =========================================
# Cell 26. 자연어 질의 응답 결과 출력
# =========================================

if "final_state" not in globals() or not final_state:
    print("아직 final_state가 없습니다. Cell 22 또는 Cell 27에서 파이프라인을 먼저 실행하세요.")
else:
    response = build_natural_language_response(final_state, top_n=10)
    print(response)


========== 자연어 질의 응답 ==========
사용자 질의: 오늘일자 기준으로 조중동의 건설경기 관련 뉴스 기사를 탐색해줘
기준일자: 2026-05-15
대상 언론사: 조선일보, 중앙일보, 동아일보
검색 키워드: 건설경기, 건설수주, 미분양, PF 부실, 공사비 상승, 건설투자, 주택시장

수집 후보 수: 60
접근 가능 URL 수: 60
실제 기사 수: 23
본문 확인 기사 수: 23
NVIDIA NIM 판단 성공 수: 15
Chroma 저장 수: 6

상위 관련 기사 목록 Top 10:

1. [관련] 국토부, 주택건설업계와 '타운홀 미팅'… 김윤덕 장관
   - 언론사: 조선일보
   - 작성일자: 작성일자 미확인
   - 작성자: 조선비즈
   - 주요 위험 이벤트: 공사비 상승
   - 요약: 국토부는 주택건설업계와 타운홀 미팅을 개최하여 주택 공급을 정상화하기 위한 해법을 모색하고, 공사비 상승 및 자재 수급 문제 등 애로사항을 논의했다.
   - 판단 방식: nvidia_nim
   - URL: https://biz.chosun.com/real_estate/real_estate_general/2026/05/14/W5FLP2SEUVDSRE4E7RSX6E5MMU/?outputType=amp

2. [관련] 경제부총리
   - 언론사: 조선일보
   - 작성일자: 2026-05-14
   - 작성자: 조선비즈
   - 주요 위험 이벤트: 착공 지연
   - 요약: 경제부총리가 태릉 골프장 부지에 주택 공급을 1년 앞당겨 2029년 착공하기로 결정
   - 판단 방식: nvidia_nim
   - URL: https://biz.chosun.com/policy/policy_sub/2026/05/15/32X5GGT75BF5JNJQ5NNY47RJIE/

3. [부분 관련] 금호건설, 1분기 영업익 121억… 전년比 111.2% 증가
   - 언론사: 조선일보
   - 작성일자: 2026-05-15
   - 작성자: 이성은 기자
   - 주요 위험 이

## Cell 27. 여러 자연어 질의 시뮬레이션 테스트 셀

목적:
- 사용자가 `QUERY_INDEX`만 바꿔 여러 자연어 질의의 파싱 또는 전체 실행을 테스트한다.

입력값:
- `TEST_QUERIES`
- `QUERY_INDEX`
- `RUN_FULL_PIPELINE`

출력값:
- False 모드: 자연어 질의 파싱 결과
- True 모드: 전체 Agent 실행 결과와 `final_state`

다음 셀과의 연결 관계:
- True 모드로 실행한 뒤 Cell 23~26에서 결과를 확인할 수 있다.

사용자가 수정해도 되는 부분:
- `QUERY_INDEX`
- `RUN_FULL_PIPELINE`
- TEST_QUERIES

주의사항:
- 비용 절감을 위해 기본값은 `RUN_FULL_PIPELINE = False`다.
- True로 바꾸면 RSS/URL/NIM/SQLite/Chroma/Excel까지 전체 실행된다.


In [ ]:
# =========================================
# Cell 27. 여러 자연어 질의 시뮬레이션 테스트 셀
# =========================================

TEST_QUERIES = [
    "오늘일자 기준으로 조중동의 건설경기 관련 뉴스 기사를 탐색해줘",
    "2026년 5월 13일 기준으로 조선일보와 동아일보의 미분양 관련 뉴스를 찾아줘",
    "오늘 기준으로 중앙일보의 PF 부실과 공사비 상승 관련 기사를 수집해서 검증해줘",
    "어제 기준으로 조중동의 주택시장 위험 관련 뉴스를 찾아줘",
    "2026-05-13 기준으로 조중동에서 건설수주와 건설투자 부진 관련 기사를 찾아줘",
]

QUERY_INDEX = 0
RUN_FULL_PIPELINE = True

USER_NEWS_QUERY = TEST_QUERIES[QUERY_INDEX]
print("선택 질의:", USER_NEWS_QUERY)

USER_NEWS_QUERY_CONFIG = parse_user_news_query(USER_NEWS_QUERY, use_nim_parser=True)
print("\n========== 자연어 질의 파싱 결과 ==========")
print(json.dumps(USER_NEWS_QUERY_CONFIG, ensure_ascii=False, indent=2))

if RUN_FULL_PIPELINE:
    final_state = run_news_agent_from_query(USER_NEWS_QUERY, run_excel=True, top_n=10)
else:
    print("\nRUN_FULL_PIPELINE=False 이므로 파싱 결과만 출력했습니다.")
    print("전체 Agent를 실행하려면 RUN_FULL_PIPELINE=True로 변경하세요.")


선택 질의: 오늘일자 기준으로 조중동의 건설경기 관련 뉴스 기사를 탐색해줘

========== 자연어 질의 파싱 결과 ==========
{
  "parser": "nvidia_nim",
  "original_query": "오늘일자 기준으로 조중동의 건설경기 관련 뉴스 기사를 탐색해줘",
  "target_date": "2026-05-15",
  "selected_outlets": [
    "조선일보",
    "중앙일보",
    "동아일보"
  ],
  "outlets": [
    {
      "outlet_name": "조선일보",
      "site_domains": [
        "chosun.com",
        "biz.chosun.com"
      ],
      "allowed_domains": [
        "chosun.com",
        "biz.chosun.com",
        "realty.chosun.com",
        "cbiz.chosun.com"
      ]
    },
    {
      "outlet_name": "중앙일보",
      "site_domains": [
        "joongang.co.kr"
      ],
      "allowed_domains": [
        "joongang.co.kr",
        "www.joongang.co.kr"
      ]
    },
    {
      "outlet_name": "동아일보",
      "site_domains": [
        "donga.com"
      ],
      "allowed_domains": [
        "donga.com",
        "www.donga.com"
      ]
    }
  ],
  "keywords": [
    "건설경기",
    "건설수주",
    "미분양",
    "PF 부실",
    "공사비 상승",
    "건설투자",
    "주택